# 03.2 — Etiquetado rápido mediante API

Ejecuta la taxonomía peruana mediante la API de DeepSeek. Por defecto usa `deepseek-v4-flash` para producción por su relación rapidez/precio y `deepseek-v4-pro` para la revisión de casos difíciles. El flujo conserva JSON Schema, validación semántica, reintentos, guardado incremental y reanudación por `chunk_id`.

> El modo inicial es `validate`: verifica archivos, credencial y acceso al servicio sin etiquetar la muestra. La clave API nunca debe escribirse dentro del cuaderno.

In [22]:
# Dependencias reproducibles del cuaderno.
from pathlib import Path
_req_candidates = [Path('requirements.txt'), Path('03_2_etiquetado_llm_api/requirements.txt')]
_requirements = next((p.resolve() for p in _req_candidates if p.exists()), None)
if _requirements is None:
    raise FileNotFoundError('No se encontró requirements.txt del módulo 03_2_etiquetado_llm_api')
%pip install -q -r {_requirements}

Note: you may need to restart the kernel to use updated packages.


## 1. Configuración

Todos los controles de uso están al comienzo de la siguiente celda: `RUN_MODE`, tamaños, semilla, modelos, concurrencia y tamaño de lote. La credencial se obtiene exclusivamente de `DEEPSEEK_API_KEY`.

In [ ]:
from __future__ import annotations

from collections import Counter, defaultdict
from concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, wait
from datetime import datetime
from pathlib import Path
import hashlib
import json
import os
import random
import re
import shutil
import threading
import time

import jsonschema
import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from dotenv import load_dotenv
from IPython.display import display
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score
from sklearn.preprocessing import MultiLabelBinarizer
from tqdm.auto import tqdm

# ========= AJUSTES DE EJECUCIÓN (edita normalmente solo este bloque) =========
RUN_MODE = os.getenv('ETIQUETADO_RUN_MODE', 'review').strip().lower()
# Modos: 'validate', 'pilot', 'production' o 'review'.
PILOT_SAMPLE_SIZE = 300          # Cantidad del piloto.
PRODUCTION_SAMPLE_SIZE = None   # Cantidad de producción; None = corpus completo.
REVIEW_SAMPLE_SIZE = None        # Máximo a revisar; None = todos los seleccionados.
SAMPLE_SEED = 42                 # Misma semilla = misma submuestra y reanudación.
# ============================================================================

RUN_MODES = {'validate', 'pilot', 'production', 'review'}
if RUN_MODE not in RUN_MODES:
    raise ValueError(f'ETIQUETADO_RUN_MODE debe ser uno de {sorted(RUN_MODES)}.')
for size_name, size_value in {
    'PILOT_SAMPLE_SIZE': PILOT_SAMPLE_SIZE,
    'PRODUCTION_SAMPLE_SIZE': PRODUCTION_SAMPLE_SIZE,
    'REVIEW_SAMPLE_SIZE': REVIEW_SAMPLE_SIZE,
}.items():
    if size_value is not None and (
        isinstance(size_value, bool) or not isinstance(size_value, int) or size_value < 1
    ):
        raise ValueError(f'{size_name} debe ser un entero positivo o None.')
if isinstance(SAMPLE_SEED, bool) or not isinstance(SAMPLE_SEED, int):
    raise ValueError('SAMPLE_SEED debe ser un entero.')

EJECUTAR_PILOTO = RUN_MODE == 'pilot'
EJECUTAR_PRODUCCION = RUN_MODE == 'production'
EJECUTAR_REVISION = RUN_MODE == 'review'

def encontrar_raiz(inicio: Path | None = None) -> Path:
    inicio = (inicio or Path.cwd()).resolve()
    for candidato in [inicio, *inicio.parents]:
        if (candidato / 'datos' / 'processed' / 'chunks_para_etiquetar.jsonl').exists():
            return candidato
    raise FileNotFoundError('No se encontró la raíz del proyecto desde ' + str(inicio))

ROOT = encontrar_raiz()
MODULE_DIR = ROOT / '03_2_etiquetado_llm_api'
PROCESSED_DIR = ROOT / 'datos' / 'processed'
OUTPUT_DIR = ROOT / 'datos' / 'etiquetado' / 'llm_api'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
load_dotenv(MODULE_DIR / '.env', override=False)

CHUNKS_FILE = PROCESSED_DIR / 'chunks_para_etiquetar.jsonl'
TAXONOMY_FILE = PROCESSED_DIR / 'taxonomia_moderacion.csv'
SKILL_FILE = ROOT / 'modelos' / 'skills' / 'clasificacion_moderacion_peru.md'
OPERATIVE_PROMPT_FILE = ROOT / 'para_equiquetado_LLM' / 'PROMPT_ETIQUETADO_LLM.md'
COMPACT_PROMPT_FILE = MODULE_DIR / 'prompt_operacional_compacto.md'
REFERENCE_GLOB = 'cgt_labeled_chunks_parte_*.jsonl'

API_PROVIDER = 'deepseek'
API_BASE = os.getenv('DEEPSEEK_BASE_URL', 'https://api.deepseek.com').rstrip('/')
API_KEY_ENV = 'DEEPSEEK_API_KEY'
API_KEY = os.getenv(API_KEY_ENV, '').strip()
PROMPT_MODE = 'compact'  # 'compact' para producción; 'full' solo para auditorías pequeñas
PROMPT_BUNDLE_VERSION = '1.1'
PRIMARY_MODEL_ID = os.getenv('DEEPSEEK_PRIMARY_MODEL', 'deepseek-v4-flash').strip()
PRIMARY_ANNOTATOR_ID = 'DSF'
REVIEW_MODEL_ID = os.getenv('DEEPSEEK_REVIEW_MODEL', 'deepseek-v4-pro').strip()
REVIEW_ANNOTATOR_ID = 'DSP'

TEMPERATURE = 0.0
MAX_TOKENS_PER_RECORD = 512
MAX_TOKENS_OVERHEAD = 64
MAX_TOKENS_RETRY_MULTIPLIER = 2
REQUEST_TIMEOUT_SECONDS = 180
MAX_RETRIES = 5
MAX_WORKERS = 32             # Reduce a 8–16 si tu cuenta devuelve HTTP 429.
BATCH_SIZE = 5               # El progreso se reporta en chunks, no en llamadas.
BACKOFF_BASE_SECONDS = 1.0
BACKOFF_MAX_SECONDS = 30.0
SAFE_CONTROL_RATE = 0.10
LIVE_METRICS = True
PERSIST_METRICS = True
QUARANTINE_INVALID_PROGRESS = True  # Respalda y reenvía filas antiguas incompatibles.
# USD por millón de tokens, según la tabla oficial vigente al crear este cuaderno.
MODEL_PRICING_USD_PER_MILLION = {
    'deepseek-v4-flash': {'cache_hit': 0.0028, 'cache_miss': 0.14, 'output': 0.28},
    'deepseek-v4-pro': {'cache_hit': 0.003625, 'cache_miss': 0.435, 'output': 0.87},
}

print('Raíz:', ROOT)
print('Salidas:', OUTPUT_DIR)
print('Proveedor/API:', API_PROVIDER, API_BASE)
print('Modelos:', {'primary': PRIMARY_MODEL_ID, 'review': REVIEW_MODEL_ID})
print('Paralelismo:', {'workers': MAX_WORKERS, 'batch_size': BATCH_SIZE})
print('Modo de ejecución:', RUN_MODE)
print(
    'Cantidades configuradas:',
    {'pilot': PILOT_SAMPLE_SIZE, 'production': PRODUCTION_SAMPLE_SIZE,
     'review': REVIEW_SAMPLE_SIZE},
)
print('Semilla de muestreo:', SAMPLE_SEED)

Raíz: D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4
Salidas: D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\datos\etiquetado\llm_api
Proveedor/API: deepseek https://api.deepseek.com
Modelos: {'primary': 'deepseek-v4-flash', 'review': 'deepseek-v4-pro'}
Paralelismo: {'workers': 32, 'batch_size': 5}
Modo de ejecución: review
Cantidades configuradas: {'pilot': 300, 'production': None, 'review': 10000}
Semilla de muestreo: 42


## 2. Preflight de archivos, credencial y modelos

El preflight no etiqueta ni envía textos del corpus. Confirma los archivos locales, exige que `DEEPSEEK_API_KEY` exista y consulta `/models` para verificar la credencial y los identificadores configurados.

In [24]:
ARCHIVOS_REQUERIDOS = [
    CHUNKS_FILE, TAXONOMY_FILE, SKILL_FILE, OPERATIVE_PROMPT_FILE, COMPACT_PROMPT_FILE
]
faltantes = [str(p) for p in ARCHIVOS_REQUERIDOS if not p.exists()]
if faltantes:
    raise FileNotFoundError('Faltan archivos requeridos:\n- ' + '\n- '.join(faltantes))

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def api_headers() -> dict[str, str]:
    if not API_KEY:
        raise RuntimeError(
            f'Falta {API_KEY_ENV}. Configúrala como variable de entorno y reinicia el kernel.'
        )
    return {
        'Authorization': f'Bearer {API_KEY}',
        'Content-Type': 'application/json',
        'Accept': 'application/json',
    }

def consultar_modelos() -> list[str]:
    try:
        response = requests.get(
            f'{API_BASE}/models', headers=api_headers(), timeout=30
        )
        response.raise_for_status()
    except requests.HTTPError as exc:
        status = exc.response.status_code if exc.response is not None else '?'
        detalle = (exc.response.text if exc.response is not None else str(exc))[:500]
        raise RuntimeError(f'La API rechazó el preflight (HTTP {status}): {detalle}') from exc
    except requests.RequestException as exc:
        raise RuntimeError(f'No se pudo conectar con {API_BASE}: {exc}') from exc
    return [str(m['id']) for m in response.json().get('data', []) if m.get('id')]

modelos_visibles = consultar_modelos()
if PRIMARY_MODEL_ID not in modelos_visibles:
    raise RuntimeError(
        f'El modelo primario {PRIMARY_MODEL_ID!r} no aparece en /models: {modelos_visibles}'
    )
if RUN_MODE == 'review' and REVIEW_MODEL_ID not in modelos_visibles:
    raise RuntimeError(
        f'El modelo de revisión {REVIEW_MODEL_ID!r} no aparece en /models: {modelos_visibles}'
    )

print('Credencial y API verificadas; no se enviaron textos del corpus.')
print('Modelos visibles:', modelos_visibles or '(ninguno)')
print('SHA256 skill:', sha256_file(SKILL_FILE))
print('SHA256 prompt:', sha256_file(OPERATIVE_PROMPT_FILE))
print('SHA256 prompt compacto:', sha256_file(COMPACT_PROMPT_FILE))
print('Modo de prompt:', PROMPT_MODE)
print(
    f'API lista: primary={PRIMARY_MODEL_ID}, review={REVIEW_MODEL_ID}, '
    f'workers={MAX_WORKERS}, batch={BATCH_SIZE}'
)

Credencial y API verificadas; no se enviaron textos del corpus.
Modelos visibles: ['deepseek-v4-flash', 'deepseek-v4-pro']
SHA256 skill: 45f9d3231a92453835ee6dfcbb8cfff0b682718caa4111f4bca3e841573a0efb
SHA256 prompt: a42004317f115b52b429771214afe3491224c0a19f14c9a07ea153ed74a82a57
SHA256 prompt compacto: 52d4fec14ad433d35ec20de5f51a6954aad69dcedd1422059419dcecc2f9e778
Modo de prompt: compact
API lista: primary=deepseek-v4-flash, review=deepseek-v4-pro, workers=32, batch=5


## 3. Carga canónica y taxonomía

La entrada puede ocupar decenas de MB, pero cabe holgadamente en la RAM disponible. Se valida `chunk_id` antes de cualquier inferencia.

In [25]:
def leer_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open('r', encoding='utf-8') as f:
        for lineno, line in enumerate(f, 1):
            if not line.strip():
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f'JSON inválido en {path}, línea {lineno}: {exc}') from exc
    return rows

taxonomy_df = pd.read_csv(TAXONOMY_FILE).fillna('')
ALLOWED_FLAGS = set(taxonomy_df.loc[taxonomy_df['categoria'] == 'FLAG', 'label'])
ALLOWED_LABELS = set(taxonomy_df.loc[taxonomy_df['categoria'] != 'FLAG', 'label'])
SAFE_LABELS = {'seguro', 'seguro_ironia_marcada'}
DAMAGE_LABELS = ALLOWED_LABELS - SAFE_LABELS
LABEL_ORDER = taxonomy_df.loc[taxonomy_df['categoria'] != 'FLAG', 'label'].tolist()
FLAG_ORDER = taxonomy_df.loc[taxonomy_df['categoria'] == 'FLAG', 'label'].tolist()

chunks = leer_jsonl(CHUNKS_FILE)
chunk_ids = [r.get('chunk_id') for r in chunks]
if any(not x for x in chunk_ids):
    raise ValueError('Hay chunk_id nulos o vacíos en el canónico.')
if len(chunk_ids) != len(set(chunk_ids)):
    raise ValueError('Hay chunk_id duplicados en el canónico.')
if any(not isinstance(r.get('text'), str) or not r['text'].strip() for r in chunks):
    raise ValueError('Hay chunks sin texto válido.')

CHUNK_BY_ID = {r['chunk_id']: r for r in chunks}
CANONICAL_POSITION = {r['chunk_id']: i for i, r in enumerate(chunks)}
print(f'Chunks: {len(chunks):,}')
print(f'Etiquetas: {len(ALLOWED_LABELS)} | Flags: {len(ALLOWED_FLAGS)}')

Chunks: 69,853
Etiquetas: 14 | Flags: 3


## 4. Prompt trazable y JSON Schema

El modo compacto conserva los criterios operativos y registra los hashes de las fuentes completas; el modo full las inserta literalmente. El envoltorio `annotations` permite procesar lotes y cada elemento se transforma después en una línea JSONL con campos administrativos constantes.

In [26]:
skill_text = SKILL_FILE.read_text(encoding='utf-8')
operative_prompt_text = OPERATIVE_PROMPT_FILE.read_text(encoding='utf-8')
compact_prompt_text = COMPACT_PROMPT_FILE.read_text(encoding='utf-8')
taxonomy_text = TAXONOMY_FILE.read_text(encoding='utf-8')
if PROMPT_MODE not in {'compact', 'full'}:
    raise ValueError("PROMPT_MODE debe ser 'compact' o 'full'.")
authority_text = (
    compact_prompt_text
    if PROMPT_MODE == 'compact'
    else f'=== SKILL ===\n{skill_text}\n\n=== PROMPT OPERATIVO ===\n{operative_prompt_text}'
)

SYSTEM_PROMPT = f'''Eres el clasificador de este proyecto.
Las siguientes fuentes son la autoridad normativa completa. No uses una taxonomía externa.

=== REGLAS OPERATIVAS ({PROMPT_MODE}) ===
{authority_text}

=== TAXONOMÍA CSV ===
{taxonomy_text}

ADAPTACIÓN TÉCNICA PARA API:
- Recibirás de 1 a {BATCH_SIZE} chunks por llamada.
- Devuelve exclusivamente JSON válido con el objeto raíz annotations exigido por el JSON Schema.
- Conserva exactamente el orden y chunk_id de entrada.
- Analiza cada chunk de forma independiente.
- Usa exclusivamente las categorías y flags literales incluidos en la taxonomía.
- ironia_ambigua, humor_encubridor y contexto_necesario van solo en flags, nunca en labels.
- humor_encubridor acompaña la categoría de daño correspondiente; nunca la reemplaza.
- Si solo detectas flags pero ninguna categoría de daño, elimina esos flags y usa seguro.
- notes siempre debe ser texto: usa "" si no hay observación; máximo 140 caracteres.
- justificacion debe ser breve y no superar 450 caracteres.
- No expongas razonamiento interno; justifica brevemente el criterio aplicado.
- El programa añadirá los campos administrativos y escribirá una línea JSONL por anotación.
'''

SEMANTIC_FIELDS = {
    'chunk_id', 'labels', 'flags', 'needs_review', 'notes',
    'score_confianza', 'justificacion'
}
FINAL_FIELDS = {
    'chunk_id', 'labels', 'flags', 'needs_review', 'notes',
    'annotator_type', 'annotator_id', 'annotator_model', 'skill_file',
    'score_confianza', 'justificacion', 'annotated_at'
}

def response_schema(batch_length: int) -> dict:
    annotation = {
        'type': 'object',
        'additionalProperties': False,
        'properties': {
            'chunk_id': {'type': 'string', 'minLength': 1, 'maxLength': 160},
            'labels': {
                'type': 'array', 'minItems': 1, 'uniqueItems': True,
                'items': {'type': 'string', 'enum': sorted(ALLOWED_LABELS)},
            },
            'flags': {
                'type': 'array', 'uniqueItems': True,
                'items': {'type': 'string', 'enum': sorted(ALLOWED_FLAGS)},
            },
            'needs_review': {'type': 'boolean'},
            'notes': {'type': 'string', 'maxLength': 160},
            'score_confianza': {'type': 'number', 'minimum': 0, 'maximum': 1},
            'justificacion': {'type': 'string', 'minLength': 1, 'maxLength': 500},
        },
        'required': sorted(SEMANTIC_FIELDS),
    }
    schema = {
        'type': 'object',
        'additionalProperties': False,
        'properties': {
            'annotations': {
                'type': 'array', 'minItems': batch_length, 'maxItems': batch_length,
                'items': annotation,
            }
        },
        'required': ['annotations'],
    }
    return {
        'type': 'json_schema',
        'json_schema': {'name': 'moderacion_peru_batch', 'strict': True, 'schema': schema},
    }

## 5. Cliente, reglas de validación y reintentos

In [27]:
def validar_semantica(row: dict, expected_id: str) -> list[str]:
    errors = []
    if set(row) != SEMANTIC_FIELDS:
        errors.append(f'campos inesperados/faltantes: {sorted(set(row) ^ SEMANTIC_FIELDS)}')
    if row.get('chunk_id') != expected_id:
        errors.append(f'chunk_id esperado {expected_id!r}, recibido {row.get("chunk_id")!r}')
    labels = row.get('labels', [])
    flags = row.get('flags', [])
    if not isinstance(labels, list) or not labels:
        errors.append('labels debe ser una lista no vacía')
    elif not set(labels) <= ALLOWED_LABELS:
        errors.append('labels contiene valores fuera de la taxonomía')
    if not isinstance(flags, list) or not set(flags) <= ALLOWED_FLAGS:
        errors.append('flags contiene valores fuera de la taxonomía')
    safe = set(labels) & SAFE_LABELS
    damage = set(labels) & DAMAGE_LABELS
    if safe and damage:
        errors.append('una etiqueta segura no puede coexistir con daño')
    if len(safe) > 1:
        errors.append('seguro y seguro_ironia_marcada no deben coexistir')
    if flags and not damage:
        errors.append('los flags transversales requieren al menos una etiqueta de daño')
    score = row.get('score_confianza')
    if isinstance(score, bool) or not isinstance(score, (int, float)) or not 0 <= score <= 1:
        errors.append('score_confianza debe estar entre 0 y 1')
    else:
        if ({'ironia_ambigua', 'contexto_necesario'} & set(flags)) and score > 0.65:
            errors.append('flag ambiguo/contextual limita score_confianza a 0.65')
        if (flags or score < 0.70) and row.get('needs_review') is not True:
            errors.append('flags o score < 0.70 obligan needs_review=true')
    if not isinstance(row.get('needs_review'), bool):
        errors.append('needs_review debe ser booleano')
    if not isinstance(row.get('notes'), str):
        errors.append('notes debe ser texto')
    if not isinstance(row.get('justificacion'), str) or not row.get('justificacion', '').strip():
        errors.append('justificacion no puede estar vacía')
    return errors

def completar_fila(row: dict, model_id: str, annotator_id: str) -> dict:
    return {
        'chunk_id': row['chunk_id'],
        'labels': row['labels'],
        'flags': row['flags'],
        'needs_review': row['needs_review'],
        'notes': row['notes'],
        'annotator_type': 'llm',
        'annotator_id': annotator_id,
        'annotator_model': model_id,
        'skill_file': SKILL_FILE.name,
        'score_confianza': float(row['score_confianza']),
        'justificacion': row['justificacion'].strip(),
        'annotated_at': datetime.now().astimezone().isoformat(timespec='seconds'),
    }

def construir_entrada(records: list[dict]) -> str:
    payload = []
    for r in records:
        item = {
            'chunk_id': r['chunk_id'],
            'text': r['text'],
            'channel_title': r.get('channel_title'),
            'video_title': r.get('video_title'),
        }
        for key in ('contexto_anterior', 'contexto_posterior'):
            if r.get(key):
                item[key] = r[key]
        payload.append(item)
    return 'Clasifica estos registros según las fuentes de autoridad:\n' + json.dumps(payload, ensure_ascii=False)

_thread_local = threading.local()

def obtener_sesion_api() -> requests.Session:
    session = getattr(_thread_local, 'session', None)
    if session is None:
        session = requests.Session()
        adapter = HTTPAdapter(
            pool_connections=MAX_WORKERS, pool_maxsize=MAX_WORKERS, max_retries=0
        )
        session.mount('https://', adapter)
        session.mount('http://', adapter)
        _thread_local.session = session
    return session

def parsear_json_respuesta(content: str | dict) -> dict:
    if isinstance(content, dict):
        return content
    clean = content.strip()
    if clean.startswith('```'):
        clean = re.sub(r'^```(?:json)?\s*|\s*```$', '', clean, flags=re.IGNORECASE)
    return json.loads(clean)

def llamar_api(
    records: list[dict], model_id: str, correction: str = '', max_tokens: int | None = None
) -> tuple[list[dict], dict]:
    user_content = construir_entrada(records)
    if correction:
        user_content += '\nLa respuesta anterior fue inválida. Corrige estos errores:\n' + correction
    token_budget = max_tokens or (
        MAX_TOKENS_OVERHEAD + MAX_TOKENS_PER_RECORD * len(records)
    )
    body = {
        'model': model_id,
        'messages': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': user_content},
        ],
        'temperature': TEMPERATURE,
        'max_tokens': token_budget,
        'stream': False,
        # DeepSeek JSON Output garantiza JSON; el esquema completo se valida localmente.
        'response_format': {'type': 'json_object'},
        # Clasificación: desactivar razonamiento reduce latencia y tokens de salida.
        'thinking': {'type': 'disabled'},
    }
    response = obtener_sesion_api().post(
        f'{API_BASE}/chat/completions', headers=api_headers(), json=body,
        timeout=(15, REQUEST_TIMEOUT_SECONDS),
    )
    if response.status_code >= 400:
        retry_after = response.headers.get('Retry-After')
        detalle = response.text[:700]
        raise RuntimeError(
            f'API HTTP {response.status_code}; Retry-After={retry_after!r}; {detalle}'
        )
    response_json = response.json()
    choice = response_json['choices'][0]
    reasoning_tokens = (
        response_json.get('usage', {}).get('completion_tokens_details', {}).get('reasoning_tokens', 0)
    )
    if reasoning_tokens:
        raise RuntimeError(
            f'El modelo usó {reasoning_tokens} tokens de razonamiento pese a thinking=disabled.'
        )
    if choice.get('finish_reason') == 'length':
        raise RuntimeError(
            f'La respuesta agotó el límite de {token_budget} tokens antes de cerrar el JSON.'
        )
    content = choice['message']['content']
    if not content:
        raise RuntimeError('La API devolvió content vacío.')
    parsed = parsear_json_respuesta(content)
    if not isinstance(parsed, dict) or set(parsed) != {'annotations'}:
        raise ValueError('La raíz debe ser un objeto JSON que contenga solo annotations.')
    annotations = parsed['annotations']
    if not isinstance(annotations, list) or len(annotations) != len(records):
        raise ValueError(
            f'annotations debe contener {len(records)} elementos; '
            f'recibidos={len(annotations) if isinstance(annotations, list) else type(annotations).__name__}'
        )
    return annotations, response_json.get('usage', {})

def normalizar_semantica(row: dict) -> dict:
    """Normaliza límites mecánicos sin modificar etiquetas ni flags."""
    normalized = dict(row)
    labels = normalized.get('labels')
    flags = normalized.get('flags')
    if flags is None:
        flags = []
        normalized['flags'] = flags
    if isinstance(labels, list) and isinstance(flags, list):
        misplaced_flags = [value for value in labels if value in ALLOWED_FLAGS]
        if misplaced_flags:
            normalized['labels'] = [
                value for value in labels if value not in ALLOWED_FLAGS
            ]
            normalized['flags'] = list(dict.fromkeys([*flags, *misplaced_flags]))
    notes = normalized.get('notes')
    if notes is None:
        normalized['notes'] = ''
    elif isinstance(notes, str):
        normalized['notes'] = notes.strip()[:160]
    justificacion = normalized.get('justificacion')
    if isinstance(justificacion, str):
        normalized['justificacion'] = justificacion.strip()[:500]
    flags = normalized.get('flags')
    score = normalized.get('score_confianza')
    if isinstance(flags, list):
        if (
            {'ironia_ambigua', 'contexto_necesario'} & set(flags)
            and isinstance(score, (int, float)) and not isinstance(score, bool)
            and score > 0.65
        ):
            normalized['score_confianza'] = 0.65
            score = 0.65
        if flags:
            normalized['needs_review'] = True
    if (
        isinstance(score, (int, float)) and not isinstance(score, bool)
        and score < 0.70
    ):
        normalized['needs_review'] = True
    return normalized

def acumular_uso(target: defaultdict, usage: dict) -> None:
    for key, value in usage.items():
        if isinstance(value, (int, float)) and not isinstance(value, bool):
            target[key] += value

def clasificar_lote(records: list[dict], model_id: str, annotator_id: str) -> tuple[list[dict], dict]:
    if not model_id:
        raise ValueError('Debes configurar el identificador del modelo.')
    if not re.fullmatch(r'[A-Z0-9]{3}', annotator_id):
        raise ValueError('annotator_id debe tener exactamente tres caracteres A-Z/0-9.')
    original_ids = [r['chunk_id'] for r in records]
    pending = list(records)
    completed = {}
    usage_total = defaultdict(int)
    correction = ''
    last_error = None
    for attempt in range(1, MAX_RETRIES + 2):
        base_token_budget = MAX_TOKENS_OVERHEAD + MAX_TOKENS_PER_RECORD * len(pending)
        token_budget = base_token_budget * (MAX_TOKENS_RETRY_MULTIPLIER ** (attempt - 1))
        try:
            annotations, usage = llamar_api(
                pending, model_id, correction, max_tokens=token_budget
            )
            acumular_uso(usage_total, usage)
            expected = [r['chunk_id'] for r in pending]
            received = [r.get('chunk_id') if isinstance(r, dict) else None for r in annotations]
            if received != expected:
                raise ValueError(f'orden/IDs incorrectos: esperado={expected}, recibido={received}')
            next_pending = []
            all_errors = []
            item_schema = (
                response_schema(1)['json_schema']['schema']
                ['properties']['annotations']['items']
            )
            for record, row in zip(pending, annotations):
                expected_id = record['chunk_id']
                if not isinstance(row, dict):
                    normalized = {}
                    row_errors = ['la anotación no es un objeto JSON']
                else:
                    normalized = normalizar_semantica(row)
                    if normalized.get('labels') == []:
                        active_flags = normalized.get('flags') or []
                        row_errors = [
                            'labels quedó vacío. Los flags transversales '
                            f'{active_flags} nunca sustituyen la categoría principal. '
                            'Vuelve a analizar: si existe daño, añade en labels una o más '
                            'categorías de daño y conserva los flags; si no existe daño, '
                            'elimina los flags y usa labels=["seguro"].'
                        ]
                    else:
                        try:
                            jsonschema.validate(normalized, item_schema)
                        except jsonschema.ValidationError as exc:
                            row_errors = [f'JSON Schema: {exc.message}']
                        else:
                            row_errors = validar_semantica(normalized, expected_id)
                if row_errors:
                    next_pending.append(record)
                    all_errors.extend(f'{expected_id}: {error}' for error in row_errors)
                else:
                    completed[expected_id] = completar_fila(
                        normalized, model_id, annotator_id
                    )
            if not next_pending:
                return [completed[cid] for cid in original_ids], dict(usage_total)
            pending = next_pending
            last_error = ValueError('; '.join(all_errors))
            correction = str(last_error)[:3000]
        except Exception as exc:
            last_error = exc
            correction = str(exc)[:3000]
        print(
            f'Intento {attempt}/{MAX_RETRIES + 1} inválido; '
            f'se reintentan {len(pending)} registro(s): {correction[:500]}'
        )
        if attempt <= MAX_RETRIES:
            delay = min(
                BACKOFF_MAX_SECONDS, BACKOFF_BASE_SECONDS * 2 ** (attempt - 1)
            )
            time.sleep(delay + random.random() * min(1.0, delay * 0.25))
    pending_ids = [r['chunk_id'] for r in pending]
    raise RuntimeError(
        f'Fallaron {pending_ids} después de {MAX_RETRIES + 1} intentos: {last_error}'
    ) from last_error

## 6. Muestra piloto reproducible

Incluye primero los IDs de la referencia existente y completa la muestra mediante recorrido balanceado por canal. Esto sirve para comparar modelos; no asigna etiquetas mediante heurísticas.

In [28]:
reference_files = sorted((ROOT / 'para_equiquetado_LLM').glob(REFERENCE_GLOB))
reference_rows = [row for path in reference_files for row in leer_jsonl(path)]
REFERENCE_BY_ID = {r['chunk_id']: r for r in reference_rows if r.get('chunk_id') in CHUNK_BY_ID}

def seleccionar_submuestra_aleatoria(
    records: list[dict], n: int | None, seed: int
) -> list[dict]:
    """Crea un orden aleatorio reproducible y devuelve su prefijo de tamaño n."""
    shuffled = list(records)
    random.Random(seed).shuffle(shuffled)
    return shuffled if n is None else shuffled[:min(n, len(shuffled))]

def seleccionar_piloto(records: list[dict], n: int, seed: int, priority_ids: set[str]) -> list[dict]:
    n = min(n, len(records))
    selected_ids = []
    selected_set = set()
    for r in records:
        if r['chunk_id'] in priority_ids and len(selected_ids) < n:
            selected_ids.append(r['chunk_id'])
            selected_set.add(r['chunk_id'])
    groups = defaultdict(list)
    for r in records:
        if r['chunk_id'] not in selected_set:
            groups[r.get('channel_title') or '__sin_canal__'].append(r['chunk_id'])
    rng = random.Random(seed)
    for ids in groups.values():
        rng.shuffle(ids)
    channels = sorted(groups)
    while len(selected_ids) < n and channels:
        next_channels = []
        for channel in channels:
            if groups[channel] and len(selected_ids) < n:
                cid = groups[channel].pop()
                selected_ids.append(cid)
                selected_set.add(cid)
            if groups[channel]:
                next_channels.append(channel)
        channels = next_channels
    selected_ids.sort(key=CANONICAL_POSITION.get)
    return [CHUNK_BY_ID[cid] for cid in selected_ids]

pilot_rows = seleccionar_piloto(
    chunks, PILOT_SAMPLE_SIZE, SAMPLE_SEED, set(REFERENCE_BY_ID)
)
pilot_manifest = OUTPUT_DIR / f'piloto_ids_n{len(pilot_rows)}_seed{SAMPLE_SEED}.json'
manifest = {
    'api_provider': API_PROVIDER,
    'api_base': API_BASE,
    'model': PRIMARY_MODEL_ID,
    'annotator_id': PRIMARY_ANNOTATOR_ID,
    'prompt_bundle_version': PROMPT_BUNDLE_VERSION,
    'batch_size': BATCH_SIZE,
    'max_workers': MAX_WORKERS,
    'seed': SAMPLE_SEED,
    'n': len(pilot_rows),
    'canonical_sha256': sha256_file(CHUNKS_FILE),
    'skill_sha256': sha256_file(SKILL_FILE),
    'operative_prompt_sha256': sha256_file(OPERATIVE_PROMPT_FILE),
    'compact_prompt_sha256': sha256_file(COMPACT_PROMPT_FILE),
    'prompt_mode': PROMPT_MODE,
    'chunk_ids': [r['chunk_id'] for r in pilot_rows],
}
pilot_manifest.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'Piloto: {len(pilot_rows)} chunks; referencia coincidente: {len(REFERENCE_BY_ID)}')
print('Manifiesto:', pilot_manifest)

Piloto: 300 chunks; referencia coincidente: 60
Manifiesto: D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\datos\etiquetado\llm_api\piloto_ids_n300_seed42.json


## 7. Ejecutor incremental y reanudable

No reintenta IDs ya guardados después de validar el archivo existente. Ejecuta solicitudes concurrentes reutilizando conexiones HTTPS, pero valida y escribe los resultados desde el hilo principal siguiendo el orden reproducible de la selección. Cada lote se fuerza a disco y actualiza un panel de métricas y un archivo lateral `*.metrics.json`. Ante HTTP 429 o errores transitorios aplica reintentos con espera exponencial.

In [29]:
def slug_model(model_id: str) -> str:
    return re.sub(r'[^a-zA-Z0-9._-]+', '_', model_id).strip('_').lower() or 'modelo_sin_id'

def validar_fila_final(row: dict, canonical_ids: set[str]) -> list[str]:
    errors = []
    if set(row) != FINAL_FIELDS:
        errors.append(f'campos inesperados/faltantes: {sorted(set(row) ^ FINAL_FIELDS)}')
    cid = row.get('chunk_id')
    if cid not in canonical_ids:
        errors.append(f'chunk_id ajeno al canónico: {cid!r}')
    semantic = {key: row.get(key) for key in SEMANTIC_FIELDS}
    errors.extend(validar_semantica(semantic, cid))
    if row.get('annotator_type') != 'llm':
        errors.append('annotator_type debe ser llm')
    if row.get('skill_file') != SKILL_FILE.name:
        errors.append('skill_file incorrecto')
    return errors

def cargar_progreso(path: Path, allowed_ids: set[str]) -> tuple[list[dict], set[str]]:
    if not path.exists():
        return [], set()
    existing = leer_jsonl(path)
    ids = [r.get('chunk_id') for r in existing]
    if len(ids) != len(set(ids)):
        raise ValueError(f'Hay chunk_id duplicados en {path}. No se reanudará.')
    invalid_rows = []
    valid_rows = []
    for i, row in enumerate(existing, 1):
        errors = validar_fila_final(row, allowed_ids)
        if errors:
            invalid_rows.append({
                'line_number': i, 'chunk_id': row.get('chunk_id'),
                'errors': errors, 'row': row,
            })
        else:
            valid_rows.append(row)
    if invalid_rows:
        if not QUARANTINE_INVALID_PROGRESS:
            first = invalid_rows[0]
            raise ValueError(
                f'Fila existente inválida {first["line_number"]} en {path}: '
                f'{first["errors"]}'
            )
        stamp = datetime.now().strftime('%Y%m%dT%H%M%S')
        backup_path = path.with_name(f'{path.stem}.backup_{stamp}{path.suffix}')
        report_path = path.with_name(f'{path.stem}.invalid_rows_{stamp}.json')
        shutil.copy2(path, backup_path)
        report_path.write_text(
            json.dumps(invalid_rows, ensure_ascii=False, indent=2), encoding='utf-8'
        )
        temp_path = path.with_suffix(path.suffix + '.tmp')
        with temp_path.open('w', encoding='utf-8', newline='\n') as f:
            for row in valid_rows:
                f.write(json.dumps(row, ensure_ascii=False, separators=(',', ':')) + '\n')
        os.replace(temp_path, path)
        print(
            f'Se retiraron {len(invalid_rows)} filas incompatibles para reetiquetarlas. '
            f'Respaldo: {backup_path.name}; reporte: {report_path.name}'
        )
        existing = valid_rows
        ids = [r['chunk_id'] for r in existing]
    return existing, set(ids)

def acumular_metricas(state: dict, rows: list[dict]) -> None:
    for row in rows:
        labels = set(row['labels'])
        flags = set(row['flags'])
        state['label_counts'].update(labels)
        state['flag_counts'].update(flags)
        state['completed'] += 1
        state['safe_chunks'] += int(bool(labels & SAFE_LABELS))
        state['damage_chunks'] += int(bool(labels & DAMAGE_LABELS))
        state['needs_review'] += int(bool(row['needs_review']))
        state['confidence_sum'] += float(row['score_confianza'])

def crear_estado_metricas(existing_rows: list[dict]) -> dict:
    state = {
        'label_counts': Counter(),
        'flag_counts': Counter(),
        'completed': 0,
        'safe_chunks': 0,
        'damage_chunks': 0,
        'needs_review': 0,
        'confidence_sum': 0.0,
    }
    acumular_metricas(state, existing_rows)
    return state

def resumen_metricas(state: dict, total_target: int) -> dict:
    completed = state['completed']
    return {
        'completed': completed,
        'pending': max(total_target - completed, 0),
        'progress_pct': round(100 * completed / total_target, 3) if total_target else 100.0,
        'safe_chunks': state['safe_chunks'],
        'damage_chunks': state['damage_chunks'],
        'needs_review': state['needs_review'],
        'needs_review_pct': round(100 * state['needs_review'] / completed, 3) if completed else 0.0,
        'mean_confidence': round(state['confidence_sum'] / completed, 4) if completed else None,
        'label_counts': {label: int(state['label_counts'][label]) for label in LABEL_ORDER},
        'flag_counts': {flag: int(state['flag_counts'][flag]) for flag in FLAG_ORDER},
    }

def tabla_metricas(state: dict, total_target: int) -> pd.DataFrame:
    summary = resumen_metricas(state, total_target)
    completed = summary['completed']
    rows = [
        {'grupo': 'PROGRESO', 'metrica': 'completados', 'valor': completed, 'porcentaje': summary['progress_pct']},
        {'grupo': 'PROGRESO', 'metrica': 'pendientes', 'valor': summary['pending'], 'porcentaje': round(100 - summary['progress_pct'], 3)},
        {'grupo': 'CLASIFICACION', 'metrica': 'chunks_seguro', 'valor': summary['safe_chunks'], 'porcentaje': round(100 * summary['safe_chunks'] / completed, 3) if completed else 0.0},
        {'grupo': 'CLASIFICACION', 'metrica': 'chunks_con_dano', 'valor': summary['damage_chunks'], 'porcentaje': round(100 * summary['damage_chunks'] / completed, 3) if completed else 0.0},
        {'grupo': 'CALIDAD', 'metrica': 'needs_review', 'valor': summary['needs_review'], 'porcentaje': summary['needs_review_pct']},
        {'grupo': 'CALIDAD', 'metrica': 'confianza_media', 'valor': summary['mean_confidence'], 'porcentaje': None},
    ]
    for label in LABEL_ORDER:
        count = summary['label_counts'][label]
        rows.append({
            'grupo': 'ETIQUETA', 'metrica': label, 'valor': count,
            'porcentaje': round(100 * count / completed, 3) if completed else 0.0,
        })
    for flag in FLAG_ORDER:
        count = summary['flag_counts'][flag]
        rows.append({
            'grupo': 'FLAG', 'metrica': flag, 'valor': count,
            'porcentaje': round(100 * count / completed, 3) if completed else 0.0,
        })
    return pd.DataFrame(rows)

def guardar_metricas(
    output_path: Path, state: dict, total_target: int, model_id: str, annotator_id: str,
) -> Path:
    metrics_path = output_path.with_suffix('.metrics.json')
    payload = {
        'source_output': str(output_path),
        'model': model_id,
        'annotator_id': annotator_id,
        'prompt_mode': PROMPT_MODE,
        'prompt_bundle_version': PROMPT_BUNDLE_VERSION,
        'skill_sha256': sha256_file(SKILL_FILE),
        'operative_prompt_sha256': sha256_file(OPERATIVE_PROMPT_FILE),
        'compact_prompt_sha256': sha256_file(COMPACT_PROMPT_FILE),
        'updated_at': datetime.now().astimezone().isoformat(timespec='seconds'),
        **resumen_metricas(state, total_target),
    }
    temp_path = metrics_path.with_suffix(metrics_path.suffix + '.tmp')
    temp_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    os.replace(temp_path, metrics_path)
    return metrics_path

def actualizar_panel_metricas(handle, state: dict, total_target: int):
    if not LIVE_METRICS:
        return handle
    table = tabla_metricas(state, total_target)
    if handle is None:
        return display(table, display_id=True)
    handle.update(table)
    return handle

def estimar_costo_usd(usage: dict, model_id: str) -> float | None:
    prices = MODEL_PRICING_USD_PER_MILLION.get(model_id)
    if not prices:
        return None
    prompt_total = int(usage.get('prompt_tokens', 0) or 0)
    cache_hit = int(usage.get('prompt_cache_hit_tokens', 0) or 0)
    cache_miss = int(
        usage.get('prompt_cache_miss_tokens', max(prompt_total - cache_hit, 0)) or 0
    )
    completion = int(usage.get('completion_tokens', 0) or 0)
    cost = (
        cache_hit * prices['cache_hit']
        + cache_miss * prices['cache_miss']
        + completion * prices['output']
    ) / 1_000_000
    return round(cost, 6)

def ejecutar_etiquetado(
    records: list[dict], output_path: Path, model_id: str, annotator_id: str,
    batch_size: int = BATCH_SIZE, limit: int | None = None,
    max_workers: int = MAX_WORKERS,
) -> dict:
    if batch_size < 1 or max_workers < 1:
        raise ValueError('batch_size y max_workers deben ser mayores que cero.')
    allowed_ids = {r['chunk_id'] for r in records}
    existing, completed_ids = cargar_progreso(output_path, allowed_ids)
    total_target = len(records)
    metrics_state = crear_estado_metricas(existing)
    metrics_handle = actualizar_panel_metricas(None, metrics_state, total_target)
    metrics_path = None
    if PERSIST_METRICS:
        metrics_path = guardar_metricas(
            output_path, metrics_state, total_target, model_id, annotator_id
        )
    pending = [r for r in records if r['chunk_id'] not in completed_ids]
    if limit is not None:
        pending = pending[:limit]
    if not pending:
        print('No hay registros pendientes para esta corrida.')
        return {
            'new_rows': 0, 'elapsed_seconds': 0, 'chunks_per_minute': None,
            'metrics_file': str(metrics_path) if metrics_path else None,
            **resumen_metricas(metrics_state, total_target),
        }
    started = time.perf_counter()
    usage_total = defaultdict(int)
    new_count = 0
    batches = [
        pending[start:start + batch_size]
        for start in range(0, len(pending), batch_size)
    ]
    total_batches = len(batches)
    with (
        output_path.open('a', encoding='utf-8', newline='\n') as f,
        ThreadPoolExecutor(max_workers=max_workers) as executor,
        tqdm(
            total=total_target, initial=len(existing), unit='chunk',
            desc=output_path.stem
        ) as progress,
    ):
        in_flight = {}
        ready = {}
        next_submit = 0
        next_write = 0
        while next_submit < min(max_workers, total_batches):
            future = executor.submit(
                clasificar_lote, batches[next_submit], model_id, annotator_id
            )
            in_flight[future] = next_submit
            next_submit += 1
        while in_flight:
            done, _ = wait(in_flight, return_when=FIRST_COMPLETED)
            for future in done:
                batch_index = in_flight.pop(future)
                ready[batch_index] = future.result()
                if next_submit < total_batches:
                    next_future = executor.submit(
                        clasificar_lote, batches[next_submit], model_id, annotator_id
                    )
                    in_flight[next_future] = next_submit
                    next_submit += 1
            while next_write in ready:
                rows, usage = ready.pop(next_write)
                for row in rows:
                    errors = validar_fila_final(row, allowed_ids)
                    if errors:
                        raise ValueError(
                            f'Salida final inválida para {row.get("chunk_id")}: {errors}'
                        )
                    f.write(
                        json.dumps(row, ensure_ascii=False, separators=(',', ':')) + '\n'
                    )
                    new_count += 1
                f.flush()
                os.fsync(f.fileno())
                acumular_metricas(metrics_state, rows)
                acumular_uso(usage_total, usage)
                if PERSIST_METRICS:
                    metrics_path = guardar_metricas(
                        output_path, metrics_state, total_target, model_id, annotator_id
                    )
                metrics_handle = actualizar_panel_metricas(
                    metrics_handle, metrics_state, total_target
                )
                live = resumen_metricas(metrics_state, total_target)
                progress.update(len(rows))
                progress.set_postfix(
                    completados=live['completed'], dano=live['damage_chunks'],
                    revision=live['needs_review'], confianza=live['mean_confidence'],
                )
                next_write += 1
    elapsed = time.perf_counter() - started
    stats = {
        'output': str(output_path),
        'model': model_id,
        'new_rows': new_count,
        'total_rows': len(existing) + new_count,
        'elapsed_seconds': round(elapsed, 2),
        'chunks_per_minute': round(new_count / elapsed * 60, 3) if elapsed else None,
        'usage': dict(usage_total),
        'estimated_cost_usd_new_rows': estimar_costo_usd(usage_total, model_id),
        'metrics_file': str(metrics_path) if metrics_path else None,
        **resumen_metricas(metrics_state, total_target),
    }
    print(json.dumps(stats, ensure_ascii=False, indent=2))
    return stats

## 8. Ejecutar piloto

Ejecuta primero este piloto con `deepseek-v4-flash`. La configuración inicial usa `BATCH_SIZE=5` y `MAX_WORKERS=32`; si aparecen respuestas HTTP 429, reduce los workers a 16 u 8. Ajusta `PILOT_SAMPLE_SIZE` al comienzo; una nueva ejecución omite los IDs ya guardados.

In [30]:
pilot_output = OUTPUT_DIR / f'{slug_model(PRIMARY_MODEL_ID)}_piloto.jsonl'
if EJECUTAR_PILOTO:
    pilot_stats = ejecutar_etiquetado(
        pilot_rows, pilot_output, PRIMARY_MODEL_ID, PRIMARY_ANNOTATOR_ID,
        batch_size=BATCH_SIZE, limit=None,
    )
else:
    print("Piloto no seleccionado. Usa ETIQUETADO_RUN_MODE='pilot'.")
print('Salida prevista:', pilot_output)

Piloto no seleccionado. Usa ETIQUETADO_RUN_MODE='pilot'.
Salida prevista: D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\datos\etiquetado\llm_api\deepseek-v4-flash_piloto.jsonl


## 9. Evaluación contra referencia

La referencia previa no sustituye un gold humano. Estas métricas sirven para comparar modelos bajo condiciones idénticas.

In [31]:
def evaluar_contra_referencia(prediction_file: Path, reference_by_id: dict[str, dict]) -> tuple[pd.DataFrame, dict]:
    predictions = leer_jsonl(prediction_file)
    pred_by_id = {r['chunk_id']: r for r in predictions}
    overlap_ids = [cid for cid in reference_by_id if cid in pred_by_id]
    if not overlap_ids:
        raise ValueError('No hay IDs compartidos con la referencia.')
    classes = sorted(ALLOWED_LABELS)
    mlb = MultiLabelBinarizer(classes=classes)
    mlb.fit([classes])
    y_true = mlb.transform([reference_by_id[cid]['labels'] for cid in overlap_ids])
    y_pred = mlb.transform([pred_by_id[cid]['labels'] for cid in overlap_ids])
    exact = float(np.mean([
        set(reference_by_id[cid]['labels']) == set(pred_by_id[cid]['labels']) for cid in overlap_ids
    ]))
    jaccard = []
    for cid in overlap_ids:
        a, b = set(reference_by_id[cid]['labels']), set(pred_by_id[cid]['labels'])
        jaccard.append(len(a & b) / len(a | b))
    report = classification_report(
        y_true, y_pred, target_names=classes, zero_division=0, output_dict=True
    )
    label_rows = []
    for label in classes:
        item = report[label]
        label_rows.append({
            'label': label, 'precision': item['precision'], 'recall': item['recall'],
            'f1': item['f1-score'], 'support': int(item['support']),
        })
    summary = {
        'n_overlap': len(overlap_ids),
        'exact_match': exact,
        'jaccard_mean': float(np.mean(jaccard)),
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'precision_macro': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall_macro': recall_score(y_true, y_pred, average='macro', zero_division=0),
    }
    return pd.DataFrame(label_rows), summary

if pilot_output.exists() and REFERENCE_BY_ID:
    per_label, evaluation_summary = evaluar_contra_referencia(pilot_output, REFERENCE_BY_ID)
    display(pd.DataFrame([evaluation_summary]))
    display(per_label.sort_values(['support', 'label'], ascending=[False, True]))
else:
    print('Ejecuta el piloto para calcular métricas.')

Ejecuta el piloto para calcular métricas.


## 10. Producción completa

Activa esta celda solo después de elegir el modelo con el piloto. `PRODUCTION_SAMPLE_SIZE` toma un prefijo de un orden aleatorio reproducible definido por `SAMPLE_SEED`; por ello no depende del orden del corpus. Con la misma semilla puedes aumentar la cantidad y solo se procesarán los nuevos IDs.

In [32]:
production_records = seleccionar_submuestra_aleatoria(
    chunks, PRODUCTION_SAMPLE_SIZE, SAMPLE_SEED
)
production_output = (
    OUTPUT_DIR
    / f'{slug_model(PRIMARY_MODEL_ID)}_labeled_chunks_seed{SAMPLE_SEED}.jsonl'
)
production_manifest = production_output.with_suffix('.manifest.json')
if EJECUTAR_PRODUCCION:
    production_manifest_payload = {
        'api_provider': API_PROVIDER,
        'api_base': API_BASE,
        'model': PRIMARY_MODEL_ID,
        'annotator_id': PRIMARY_ANNOTATOR_ID,
        'prompt_mode': PROMPT_MODE,
        'prompt_bundle_version': PROMPT_BUNDLE_VERSION,
        'skill_sha256': sha256_file(SKILL_FILE),
        'operative_prompt_sha256': sha256_file(OPERATIVE_PROMPT_FILE),
        'compact_prompt_sha256': sha256_file(COMPACT_PROMPT_FILE),
        'batch_size': BATCH_SIZE,
        'max_workers': MAX_WORKERS,
        'selection_method': 'deterministic_random_shuffle_prefix',
        'seed': SAMPLE_SEED,
        'requested_size': PRODUCTION_SAMPLE_SIZE,
        'selected_size': len(production_records),
        'canonical_size': len(chunks),
        'canonical_sha256': sha256_file(CHUNKS_FILE),
        'chunk_ids': [r['chunk_id'] for r in production_records],
    }
    production_manifest.write_text(
        json.dumps(production_manifest_payload, ensure_ascii=False, indent=2),
        encoding='utf-8',
    )
    production_stats = ejecutar_etiquetado(
        production_records, production_output, PRIMARY_MODEL_ID, PRIMARY_ANNOTATOR_ID,
        batch_size=BATCH_SIZE, limit=None,
    )
else:
    print("Producción no seleccionada. Usa ETIQUETADO_RUN_MODE='production'.")
print(f'Submuestra de producción: {len(production_records):,}/{len(chunks):,}')
print('Salida prevista:', production_output)
print('Manifiesto previsto:', production_manifest)

Producción no seleccionada. Usa ETIQUETADO_RUN_MODE='production'.
Submuestra de producción: 69,853/69,853
Salida prevista: D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\datos\etiquetado\llm_api\deepseek-v4-flash_labeled_chunks_seed42.jsonl
Manifiesto previsto: D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\datos\etiquetado\llm_api\deepseek-v4-flash_labeled_chunks_seed42.manifest.json


## 11. Segunda pasada independiente con contexto

Selecciona todos los casos con revisión, daño o una muestra aleatoria de seguros. El segundo modelo no ve la etiqueta previa para evitar anclaje, pero sí recibe el chunk anterior y posterior del mismo video cuando existen. Su salida se conserva como otra anotación, sin sobrescribir la primera.

In [33]:
def construir_contexto_vecino(records: list[dict]) -> dict[str, dict]:
    by_video = defaultdict(list)
    for r in records:
        by_video[r.get('video_id')].append(r)
    enriched = {}
    for video_rows in by_video.values():
        video_rows.sort(key=lambda r: (r.get('start_seconds') or 0, CANONICAL_POSITION[r['chunk_id']]))
        for i, r in enumerate(video_rows):
            item = dict(r)
            if i > 0:
                item['contexto_anterior'] = video_rows[i - 1]['text']
            if i + 1 < len(video_rows):
                item['contexto_posterior'] = video_rows[i + 1]['text']
            enriched[r['chunk_id']] = item
    return enriched

def seleccionar_revision(primary_rows: list[dict], safe_rate: float, seed: int) -> list[str]:
    rng = random.Random(seed)
    selected = []
    for row in primary_rows:
        has_damage = bool(set(row['labels']) & DAMAGE_LABELS)
        if row['needs_review'] or has_damage:
            selected.append(row['chunk_id'])
        elif set(row['labels']) <= SAFE_LABELS and rng.random() < safe_rate:
            selected.append(row['chunk_id'])
    return selected

review_output = (
    OUTPUT_DIR / (
        f'{slug_model(REVIEW_MODEL_ID)}_revision_de_'
        f'{slug_model(PRIMARY_MODEL_ID)}_seed{SAMPLE_SEED}.jsonl'
    )
    if REVIEW_MODEL_ID else None
)
review_manifest = review_output.with_suffix('.manifest.json') if review_output else None
if EJECUTAR_REVISION:
    if not REVIEW_MODEL_ID:
        raise ValueError('Configura DEEPSEEK_REVIEW_MODEL antes de ejecutar review.')
    if not production_output.exists():
        raise FileNotFoundError('No existe la primera pasada de producción.')
    primary_rows = leer_jsonl(production_output)
    review_ids = seleccionar_revision(primary_rows, SAFE_CONTROL_RATE, SAMPLE_SEED)
    enriched = construir_contexto_vecino(chunks)
    review_records = [enriched[cid] for cid in review_ids]
    if REVIEW_SAMPLE_SIZE is not None:
        review_records = review_records[:REVIEW_SAMPLE_SIZE]
    review_manifest_payload = {
        'api_provider': API_PROVIDER,
        'api_base': API_BASE,
        'source_output': str(production_output),
        'primary_model': PRIMARY_MODEL_ID,
        'review_model': REVIEW_MODEL_ID,
        'review_annotator_id': REVIEW_ANNOTATOR_ID,
        'prompt_mode': PROMPT_MODE,
        'prompt_bundle_version': PROMPT_BUNDLE_VERSION,
        'skill_sha256': sha256_file(SKILL_FILE),
        'operative_prompt_sha256': sha256_file(OPERATIVE_PROMPT_FILE),
        'compact_prompt_sha256': sha256_file(COMPACT_PROMPT_FILE),
        'seed': SAMPLE_SEED,
        'safe_control_rate': SAFE_CONTROL_RATE,
        'requested_size': REVIEW_SAMPLE_SIZE,
        'selected_size': len(review_records),
        'chunk_ids': [r['chunk_id'] for r in review_records],
    }
    review_manifest.write_text(
        json.dumps(review_manifest_payload, ensure_ascii=False, indent=2),
        encoding='utf-8',
    )
    review_stats = ejecutar_etiquetado(
        review_records, review_output, REVIEW_MODEL_ID, REVIEW_ANNOTATOR_ID,
        batch_size=BATCH_SIZE, limit=None,
    )
else:
    print("Segunda pasada no seleccionada. Usa ETIQUETADO_RUN_MODE='review'.")
print('Salida prevista:', review_output or 'configura DEEPSEEK_REVIEW_MODEL')
print('Manifiesto previsto:', review_manifest or 'configura DEEPSEEK_REVIEW_MODEL')

,grupo,metrica,valor,porcentaje
0,PROGRESO,completados,10000.0000,100.00
1,PROGRESO,pendientes,0.0000,0.00
2,CLASIFICACION,chunks_seguro,7146.0000,71.46
3,CLASIFICACION,chunks_con_dano,2854.0000,28.54
4,CALIDAD,needs_review,2089.0000,20.89
5,CALIDAD,confianza_media,0.8646,NaN
6,ETIQUETA,seguro,6758.0000,67.58
7,ETIQUETA,seguro_ironia_marcada,388.0000,3.88
8,ETIQUETA,racismo_etnico_explicito,542.0000,5.42
9,ETIQUETA,racismo_linguistico,34.0000,0.34


No hay registros pendientes para esta corrida.
Salida prevista: D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\datos\etiquetado\llm_api\deepseek-v4-pro_revision_de_deepseek-v4-flash_seed42.jsonl
Manifiesto previsto: D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\datos\etiquetado\llm_api\deepseek-v4-pro_revision_de_deepseek-v4-flash_seed42.manifest.json


## 12. Auditoría final de una salida

In [34]:
def auditar_salida(path: Path, expected_records: list[dict]) -> dict:
    rows = leer_jsonl(path)
    ids = [r.get('chunk_id') for r in rows]
    expected_ids = [r['chunk_id'] for r in expected_records]
    expected_set = set(expected_ids)
    expected_position = {cid: i for i, cid in enumerate(expected_ids)}
    errors = []
    if len(ids) != len(set(ids)):
        errors.append('chunk_id duplicados')
    for i, row in enumerate(rows, 1):
        row_errors = validar_fila_final(row, set(CHUNK_BY_ID))
        errors.extend(f'fila {i}: {e}' for e in row_errors)
    unexpected = [cid for cid in ids if cid not in expected_set]
    if unexpected:
        errors.append(f'{len(unexpected)} chunk_id fuera de la selección configurada')
    positions = [expected_position[cid] for cid in ids if cid in expected_position]
    if positions != sorted(positions):
        errors.append('el orden no coincide con la selección reproducible')
    completed_expected = len(set(ids) & expected_set)
    summary = {
        'path': str(path),
        'rows': len(rows),
        'unique_ids': len(set(ids)),
        'target_rows': len(expected_ids),
        'pending_rows': len(expected_ids) - completed_expected,
        'progress_pct': round(100 * completed_expected / len(expected_ids), 3)
        if expected_ids else 100.0,
        'needs_review': sum(bool(r.get('needs_review')) for r in rows),
        'errors': errors[:50],
        'valid': not errors,
    }
    return summary

if (
    EJECUTAR_REVISION and review_output is not None
    and review_output.exists()
):
    archivo_a_auditar, registros_esperados = review_output, review_records
elif production_output.exists():
    archivo_a_auditar, registros_esperados = production_output, production_records
else:
    archivo_a_auditar, registros_esperados = pilot_output, pilot_rows
if archivo_a_auditar.exists():
    print(json.dumps(
        auditar_salida(archivo_a_auditar, registros_esperados),
        ensure_ascii=False, indent=2,
    ))
else:
    print('Todavía no existe una salida para auditar.')

{
  "path": "D:\\trabajo_PLN\\Trabajo_PLN-MIA-Grupo4\\datos\\etiquetado\\llm_api\\deepseek-v4-pro_revision_de_deepseek-v4-flash_seed42.jsonl",
  "rows": 10000,
  "unique_ids": 10000,
  "target_rows": 10000,
  "pending_rows": 0,
  "progress_pct": 100.0,
  "needs_review": 2089,
  "errors": [],
  "valid": true
}


# 13. Validación estadística de Flash frente a Pro

## 13.1 Pregunta, diseño y alcance de la inferencia

Esta sección evalúa si las anotaciones de la primera pasada con DeepSeek Flash son consistentes con una segunda anotación independiente de DeepSeek Pro. Pro se usa como **referencia de mayor capacidad**, no como verdad de terreno: la concordancia entre modelos mide reproducibilidad, pero no sustituye una validación humana. Por ello la conclusión distingue entre (a) equivalencia para producir etiquetas finales y (b) utilidad operativa de Flash como primera pasada dentro de un flujo híbrido.

La muestra Pro no es aleatoria simple. El diseño tiene dos estratos definidos antes de observar Pro: (1) **revisión dirigida**, formada por chunks que Flash marcó con daño o revisión; y (2) **control seguro**, sorteado entre los chunks que Flash consideró seguros. Analizar sin corregir esta selección produciría sesgo de verificación (Begg & Greenes, 1983). Se emplea postestratificación con pesos de diseño $w_h=N_h/n_h$, equivalente al principio de Horvitz–Thompson cuando todas las unidades tienen igual probabilidad dentro de cada estrato (Horvitz & Thompson, 1952).

La unidad primaria es el chunk, pero varios chunks pertenecen al mismo video y no son independientes. Los intervalos de confianza se calculan mediante **bootstrap por conglomerados de video**, re-muestreando videos completos y conservando los pesos poblacionales de los estratos; este procedimiento es apropiado para datos agrupados (Field & Welsh, 2007). Se usan 5,000 réplicas y una semilla fija.

La decisión primaria reduce la taxonomía a **daño** (al menos una etiqueta de daño) frente a **seguro**. Se informan sensibilidad, especificidad, valores predictivos, exactitud balanceada, MCC, acuerdo observado, kappa de Cohen (Cohen, 1960) y AC1 de Gwet, que es más estable ante prevalencias extremas (Gwet, 2008). Para el problema multi-etiqueta se informan coincidencia exacta, Jaccard, F1 micro y F1 macro; estas métricas responden de forma diferente al desbalance de clases (Saito & Rehmsmeier, 2015; Sokolova & Lapalme, 2009).

## 13.2 Criterios definidos antes de interpretar los resultados

Los márgenes siguientes son criterios operativos del proyecto, no constantes universales. Siguen el principio de fijar por anticipado el menor efecto de interés para una prueba de equivalencia o no inferioridad (Lakens, 2017):

- Equivalencia binaria: límite superior unilateral del 95% para el desacuerdo daño/seguro menor o igual a 5 puntos porcentuales.
- Seguridad como detector: límite inferior unilateral del 95% para sensibilidad mayor o igual a 90%.
- Descarte de falsos positivos: límite inferior unilateral del 95% para especificidad mayor o igual a 95%.
- Confiabilidad del negativo: límite inferior unilateral del 95% para VPN mayor o igual a 95%.
- Balance del control aleatorio: diferencia media estandarizada absoluta y V de Cramér menores que 0.10. Se priorizan tamaños de efecto sobre p-valores, porque estos dependen fuertemente del tamaño muestral (Austin, 2009).

Que las prevalencias globales de Flash y Pro sean parecidas no demuestra acuerdo individual. La decisión de equivalencia se basa en el desacuerdo emparejado y sus intervalos, no en una prueba no significativa de diferencia de prevalencias.

In [35]:
from scipy.stats import chi2_contingency, ks_2samp

# Archivos y criterios preespecificados para esta validación.
FLASH_VALIDATION_FILE = OUTPUT_DIR / 'deepseek-v4-flash_labeled_chunks_seed42.jsonl'
PRO_VALIDATION_FILE = OUTPUT_DIR / 'deepseek-v4-pro_revision_de_deepseek-v4-flash_seed42.jsonl'
CLUSTER_BOOTSTRAP_REPS = 5_000
ANALYSIS_SEED = 20_260_725
MAX_EQUIVALENT_DISAGREEMENT = 0.05
MIN_SENSITIVITY = 0.90
MIN_SPECIFICITY = 0.95
MIN_NPV = 0.95
MAX_BALANCE_EFFECT = 0.10

for validation_path in (FLASH_VALIDATION_FILE, PRO_VALIDATION_FILE):
    if not validation_path.exists():
        raise FileNotFoundError(f'Falta la salida necesaria: {validation_path}')

flash_validation_rows = leer_jsonl(FLASH_VALIDATION_FILE)
pro_validation_rows = leer_jsonl(PRO_VALIDATION_FILE)
flash_validation_by_id = {row['chunk_id']: row for row in flash_validation_rows}
flash_validation_manifest = json.loads(
    FLASH_VALIDATION_FILE.with_suffix('.manifest.json').read_text(encoding='utf-8')
)
pro_validation_manifest = json.loads(
    PRO_VALIDATION_FILE.with_suffix('.manifest.json').read_text(encoding='utf-8')
)

def tiene_dano(row: dict) -> bool:
    return bool(set(row.get('labels', [])) & DAMAGE_LABELS)

def estrato_flash(row: dict) -> str:
    return 'dirigida' if row.get('needs_review') or tiene_dano(row) else 'control_seguro'

# Reproduce exactamente la selección documentada en la sección 11.
selection_rng = random.Random(pro_validation_manifest['seed'])
candidate_ids = []
candidate_strata = []
for row in flash_validation_rows:
    stratum = estrato_flash(row)
    if stratum == 'dirigida':
        candidate_ids.append(row['chunk_id'])
        candidate_strata.append(stratum)
    elif set(row['labels']) <= SAFE_LABELS and selection_rng.random() < pro_validation_manifest['safe_control_rate']:
        candidate_ids.append(row['chunk_id'])
        candidate_strata.append(stratum)

review_cap = pro_validation_manifest.get('requested_size')
expected_review_ids = candidate_ids if review_cap is None else candidate_ids[:review_cap]
manifest_review_ids = pro_validation_manifest['chunk_ids']
observed_review_ids = [row['chunk_id'] for row in pro_validation_rows]

# También audita que la primera pasada sea el barajado determinista declarado.
expected_flash_order = list(chunks)
random.Random(flash_validation_manifest['seed']).shuffle(expected_flash_order)
requested_flash_size = flash_validation_manifest.get('requested_size')
if requested_flash_size is not None:
    expected_flash_order = expected_flash_order[:requested_flash_size]
expected_flash_ids = [row['chunk_id'] for row in expected_flash_order]

selection_trace = {
    'flash_manifest_reproduce_barajado': expected_flash_ids == flash_validation_manifest['chunk_ids'],
    'pro_manifest_reproduce_algoritmo': expected_review_ids == manifest_review_ids,
    'salida_pro_coincide_manifiesto': observed_review_ids == manifest_review_ids,
    'ids_pro_unicos': len(observed_review_ids) == len(set(observed_review_ids)),
    'todos_los_ids_pro_existen_en_flash': set(observed_review_ids) <= set(flash_validation_by_id),
}
if not all(selection_trace.values()):
    raise AssertionError(f'Fallo de trazabilidad de la revisión: {selection_trace}')

population_strata = Counter(estrato_flash(row) for row in flash_validation_rows)
review_strata = Counter(
    estrato_flash(flash_validation_by_id[row['chunk_id']]) for row in pro_validation_rows
)
candidate_counts = Counter(candidate_strata)
selection_summary = pd.DataFrame([
    {
        'estrato': stratum,
        'N_flash': population_strata[stratum],
        'candidatos_antes_tope': candidate_counts[stratum],
        'n_revisado_pro': review_strata[stratum],
        'fraccion_final_revisada': review_strata[stratum] / population_strata[stratum],
        'peso_N_sobre_n': population_strata[stratum] / review_strata[stratum],
    }
    for stratum in ('dirigida', 'control_seguro')
])
SELECTION_SUMMARY_FILE = OUTPUT_DIR / 'validacion_flash_pro_diseno_muestral.csv'
selection_summary.to_csv(SELECTION_SUMMARY_FILE, index=False, encoding='utf-8')
print('Trazabilidad:', selection_trace)
print(f'Flash={len(flash_validation_rows):,} | Pro={len(pro_validation_rows):,} | candidatos antes del tope={len(candidate_ids):,}')
display(selection_summary)

validation_pairs = []
for pro_row in pro_validation_rows:
    flash_row = flash_validation_by_id[pro_row['chunk_id']]
    validation_pairs.append({
        'chunk_id': pro_row['chunk_id'],
        'video_id': CHUNK_BY_ID[pro_row['chunk_id']].get('video_id') or pro_row['chunk_id'],
        'stratum': estrato_flash(flash_row),
        'flash': flash_row,
        'pro': pro_row,
    })


Trazabilidad: {'flash_manifest_reproduce_barajado': True, 'pro_manifest_reproduce_algoritmo': True, 'salida_pro_coincide_manifiesto': True, 'ids_pro_unicos': True, 'todos_los_ids_pro_existen_en_flash': True}
Flash=69,853 | Pro=10,000 | candidatos antes del tope=12,032


,estrato,N_flash,candidatos_antes_tope,n_revisado_pro,fraccion_final_revisada,peso_N_sobre_n
0,dirigida,5566,5566,4646,0.834711,1.198020
1,control_seguro,64287,6466,5354,0.083283,12.007284


In [36]:
# Auditoría del componente aleatorio dentro del estrato que Flash declaró seguro.
def diferencia_media_estandarizada(a, b) -> float:
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    varianza_combinada = (a.var(ddof=1) + b.var(ddof=1)) / 2
    return 0.0 if varianza_combinada == 0 else (a.mean() - b.mean()) / np.sqrt(varianza_combinada)

def v_cramer_y_p(grupo_a, grupo_b) -> tuple[float, float]:
    categorias = sorted(set(grupo_a) | set(grupo_b), key=str)
    conteos_a, conteos_b = Counter(grupo_a), Counter(grupo_b)
    tabla = np.asarray([
        [conteos_a[categoria] for categoria in categorias],
        [conteos_b[categoria] for categoria in categorias],
    ])
    tabla = tabla[:, tabla.sum(axis=0) > 0]
    chi2, p_value, _, _ = chi2_contingency(tabla, correction=False)
    denominador = tabla.sum() * min(tabla.shape[0] - 1, tabla.shape[1] - 1)
    return np.sqrt(chi2 / denominador), p_value

control_population_ids = [
    row['chunk_id'] for row in flash_validation_rows if estrato_flash(row) == 'control_seguro'
]
reviewed_control_ids = {
    pair['chunk_id'] for pair in validation_pairs if pair['stratum'] == 'control_seguro'
}
not_reviewed_control_ids = set(control_population_ids) - reviewed_control_ids
canonical_position = {row['chunk_id']: pos for pos, row in enumerate(chunks)}

def valores_numericos(ids, extractor):
    return [float(extractor(CHUNK_BY_ID[chunk_id])) for chunk_id in ids]

numeric_extractors = {
    'caracteres_texto': lambda row: len(row.get('text') or ''),
    'palabras_texto': lambda row: len((row.get('text') or '').split()),
    'inicio_segundos': lambda row: row.get('start_seconds') or 0.0,
    'duracion_segundos': lambda row: (row.get('end_seconds') or 0.0) - (row.get('start_seconds') or 0.0),
}
balance_rows = []
for variable, extractor in numeric_extractors.items():
    seleccionados = valores_numericos(reviewed_control_ids, extractor)
    no_seleccionados = valores_numericos(not_reviewed_control_ids, extractor)
    smd = diferencia_media_estandarizada(seleccionados, no_seleccionados)
    ks = ks_2samp(seleccionados, no_seleccionados, alternative='two-sided', method='auto')
    balance_rows.append({
        'variable': variable, 'tipo': 'continua', 'tamano_efecto': abs(smd),
        'estadistico_con_signo': smd, 'p_diagnostico': ks.pvalue,
    })

categorical_extractors = {
    'canal': lambda chunk_id: CHUNK_BY_ID[chunk_id].get('channel_title') or '__sin_canal__',
    'cuartil_orden_canonico': lambda chunk_id: min(3, 4 * canonical_position[chunk_id] // len(chunks)),
    'bucket_hash_20': lambda chunk_id: int(hashlib.sha256(chunk_id.encode('utf-8')).hexdigest(), 16) % 20,
}
for variable, extractor in categorical_extractors.items():
    seleccionados = [extractor(chunk_id) for chunk_id in reviewed_control_ids]
    no_seleccionados = [extractor(chunk_id) for chunk_id in not_reviewed_control_ids]
    cramer_v, p_value = v_cramer_y_p(seleccionados, no_seleccionados)
    balance_rows.append({
        'variable': variable, 'tipo': 'categorica', 'tamano_efecto': cramer_v,
        'estadistico_con_signo': np.nan, 'p_diagnostico': p_value,
    })

balance_df = pd.DataFrame(balance_rows)
max_balance_effect_observed = balance_df['tamano_efecto'].max()
randomness_passed = all(selection_trace.values()) and max_balance_effect_observed < MAX_BALANCE_EFFECT
print(
    f'Control seguro revisado: {len(reviewed_control_ids):,}/{len(control_population_ids):,} '
    f'({len(reviewed_control_ids) / len(control_population_ids):.2%}); '
    f'máximo tamaño de efecto={max_balance_effect_observed:.4f}; balance={randomness_passed}'
)
display(balance_df)

quartile_audit = pd.crosstab(
    pd.Series([categorical_extractors['cuartil_orden_canonico'](chunk_id) + 1 for chunk_id in control_population_ids], name='cuartil'),
    pd.Series([chunk_id in reviewed_control_ids for chunk_id in control_population_ids], name='revisado_Pro'),
)
display(quartile_audit)


Control seguro revisado: 5,354/64,287 (8.33%); máximo tamaño de efecto=0.0266; balance=True


,variable,tipo,tamano_efecto,estadistico_con_signo,p_diagnostico
0,caracteres_texto,continua,0.008301,0.008301,0.261375
1,palabras_texto,continua,0.006600,0.006600,0.655795
2,inicio_segundos,continua,0.026638,0.026638,0.214686
3,duracion_segundos,continua,0.004267,0.004267,0.095301
4,canal,categorica,0.021022,NaN,0.289299
5,cuartil_orden_canonico,categorica,0.010284,NaN,0.078582
6,bucket_hash_20,categorica,0.016607,NaN,0.540501


revisado_Pro,False,True
cuartil,,
1,14475,1299
2,15192,1427
3,13997,1316
4,15269,1312


In [37]:
# Estimación postestratificada y bootstrap por video. Pro es la referencia operativa.
STRATA = ('dirigida', 'control_seguro')
STRATUM_INDEX = {stratum: index for index, stratum in enumerate(STRATA)}
BASE_FEATURES = ('n', 'tp', 'tn', 'fp', 'fn', 'exacta', 'jaccard', 'dano_pro')
N_FEATURES = len(BASE_FEATURES) + 3 * len(LABEL_ORDER)

def vector_par(pair: dict) -> np.ndarray:
    flash_labels = set(pair['flash'].get('labels', []))
    pro_labels = set(pair['pro'].get('labels', []))
    flash_damage = bool(flash_labels & DAMAGE_LABELS)
    pro_damage = bool(pro_labels & DAMAGE_LABELS)
    union = flash_labels | pro_labels
    vector = np.zeros(N_FEATURES, dtype=float)
    vector[:8] = [
        1, flash_damage and pro_damage, not flash_damage and not pro_damage,
        flash_damage and not pro_damage, not flash_damage and pro_damage,
        flash_labels == pro_labels, len(flash_labels & pro_labels) / len(union) if union else 1.0,
        pro_damage,
    ]
    for label_index, label in enumerate(LABEL_ORDER):
        offset = 8 + 3 * label_index
        vector[offset:offset + 3] = [
            label in flash_labels and label in pro_labels,
            label in flash_labels and label not in pro_labels,
            label not in flash_labels and label in pro_labels,
        ]
    return vector

video_ids = sorted({pair['video_id'] for pair in validation_pairs}, key=str)
video_index = {video_id: index for index, video_id in enumerate(video_ids)}
cluster_contributions = np.zeros((len(video_ids), len(STRATA), N_FEATURES), dtype=float)
for pair in validation_pairs:
    cluster_contributions[
        video_index[pair['video_id']], STRATUM_INDEX[pair['stratum']]
    ] += vector_par(pair)
observed_by_stratum = cluster_contributions.sum(axis=0)
population_sizes = np.asarray([population_strata[stratum] for stratum in STRATA], dtype=float)

METRIC_NAMES = [
    'accuracy_binaria', 'sensibilidad', 'especificidad', 'vpp', 'vpn',
    'coincidencia_exacta', 'jaccard', 'kappa_cohen', 'ac1_gwet', 'mcc',
    'f1_micro', 'f1_macro', 'desacuerdo_binario', 'falsos_negativos_poblacion',
    'falsos_positivos_poblacion', 'diferencia_prevalencia_flash_menos_pro',
    'exactitud_balanceada', 'dano_pro_en_control_seguro', 'sobrealerta_en_dirigida',
]

def division_segura(numerador, denominador):
    return np.nan if denominador == 0 else numerador / denominador

def estimar_metricas(suma_por_estrato: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    if np.any(suma_por_estrato[:, 0] == 0):
        return np.full(len(METRIC_NAMES), np.nan), np.full(N_FEATURES, np.nan)
    escalado = suma_por_estrato * (population_sizes / suma_por_estrato[:, 0])[:, None]
    total = escalado.sum(axis=0)
    n, tp, tn, fp, fn, exacta, jaccard, _ = total[:8]
    acuerdo = (tp + tn) / n
    sensibilidad = division_segura(tp, tp + fn)
    especificidad = division_segura(tn, tn + fp)
    vpp = division_segura(tp, tp + fp)
    vpn = division_segura(tn, tn + fn)
    prevalencia_flash = (tp + fp) / n
    prevalencia_pro = (tp + fn) / n
    acuerdo_azar = prevalencia_flash * prevalencia_pro + (1 - prevalencia_flash) * (1 - prevalencia_pro)
    kappa = division_segura(acuerdo - acuerdo_azar, 1 - acuerdo_azar)
    prevalencia_media = (prevalencia_flash + prevalencia_pro) / 2
    acuerdo_azar_ac1 = 2 * prevalencia_media * (1 - prevalencia_media)
    ac1 = division_segura(acuerdo - acuerdo_azar_ac1, 1 - acuerdo_azar_ac1)
    mcc = division_segura(
        tp * tn - fp * fn, np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    )
    label_triplets = total[8:].reshape(len(LABEL_ORDER), 3)
    tp_labels, fp_labels, fn_labels = label_triplets.T
    f1_micro = division_segura(2 * tp_labels.sum(), 2 * tp_labels.sum() + fp_labels.sum() + fn_labels.sum())
    f1_por_etiqueta = np.divide(
        2 * tp_labels, 2 * tp_labels + fp_labels + fn_labels,
        out=np.full(len(LABEL_ORDER), np.nan), where=(2 * tp_labels + fp_labels + fn_labels) > 0,
    )
    metricas = np.asarray([
        acuerdo, sensibilidad, especificidad, vpp, vpn, exacta / n, jaccard / n,
        kappa, ac1, mcc, f1_micro, np.nanmean(f1_por_etiqueta), 1 - acuerdo, fn / n, fp / n,
        prevalencia_flash - prevalencia_pro, (sensibilidad + especificidad) / 2,
        suma_por_estrato[STRATUM_INDEX['control_seguro'], 7] / suma_por_estrato[STRATUM_INDEX['control_seguro'], 0],
        suma_por_estrato[STRATUM_INDEX['dirigida'], 3] / suma_por_estrato[STRATUM_INDEX['dirigida'], 0],
    ])
    return metricas, total

point_estimates, weighted_totals = estimar_metricas(observed_by_stratum)
rng_bootstrap = np.random.default_rng(ANALYSIS_SEED)
bootstrap_estimates = np.empty((CLUSTER_BOOTSTRAP_REPS, len(METRIC_NAMES)), dtype=float)
cluster_probability = np.full(len(video_ids), 1 / len(video_ids))
for replicate in range(CLUSTER_BOOTSTRAP_REPS):
    cluster_multiplicity = rng_bootstrap.multinomial(len(video_ids), cluster_probability)
    sampled_sum = np.tensordot(cluster_multiplicity, cluster_contributions, axes=(0, 0))
    bootstrap_estimates[replicate], _ = estimar_metricas(sampled_sum)

metric_ci = pd.DataFrame({
    'metrica': METRIC_NAMES,
    'estimacion': point_estimates,
    'ic95_inferior': np.nanpercentile(bootstrap_estimates, 2.5, axis=0),
    'ic95_superior': np.nanpercentile(bootstrap_estimates, 97.5, axis=0),
    'limite_unilateral_inferior_95': np.nanpercentile(bootstrap_estimates, 5, axis=0),
    'limite_unilateral_superior_95': np.nanpercentile(bootstrap_estimates, 95, axis=0),
})

stratum_rows = []
for stratum in STRATA:
    pairs = [pair for pair in validation_pairs if pair['stratum'] == stratum]
    vectors = np.vstack([vector_par(pair) for pair in pairs]).sum(axis=0)
    n, tp, tn, fp, fn, exacta, jaccard, dano_pro = vectors[:8]
    stratum_rows.append({
        'estrato': stratum, 'n': int(n), 'acuerdo_binario': (tp + tn) / n,
        'prevalencia_dano_flash': (tp + fp) / n, 'prevalencia_dano_pro': dano_pro / n,
        'coincidencia_exacta': exacta / n, 'jaccard': jaccard / n,
    })
stratum_metrics = pd.DataFrame(stratum_rows)

label_triplets = weighted_totals[8:].reshape(len(LABEL_ORDER), 3)
per_label_rows = []
for label, (tp_label, fp_label, fn_label) in zip(LABEL_ORDER, label_triplets):
    precision = division_segura(tp_label, tp_label + fp_label)
    recall = division_segura(tp_label, tp_label + fn_label)
    per_label_rows.append({
        'etiqueta': label, 'soporte_pro_estimado': tp_label + fn_label,
        'precision': precision, 'recall': recall,
        'f1': division_segura(2 * precision * recall, precision + recall),
    })
per_label_metrics = pd.DataFrame(per_label_rows)

metric_lookup = metric_ci.set_index('metrica')
decision_table = pd.DataFrame([
    {
        'criterio': 'Equivalencia: desacuerdo binario', 'regla': 'LS unilateral 95% ≤ 0.05',
        'valor_decisivo': metric_lookup.loc['desacuerdo_binario', 'limite_unilateral_superior_95'],
        'cumple': metric_lookup.loc['desacuerdo_binario', 'limite_unilateral_superior_95'] <= MAX_EQUIVALENT_DISAGREEMENT,
    },
    {
        'criterio': 'Sensibilidad', 'regla': 'LI unilateral 95% ≥ 0.90',
        'valor_decisivo': metric_lookup.loc['sensibilidad', 'limite_unilateral_inferior_95'],
        'cumple': metric_lookup.loc['sensibilidad', 'limite_unilateral_inferior_95'] >= MIN_SENSITIVITY,
    },
    {
        'criterio': 'Especificidad', 'regla': 'LI unilateral 95% ≥ 0.95',
        'valor_decisivo': metric_lookup.loc['especificidad', 'limite_unilateral_inferior_95'],
        'cumple': metric_lookup.loc['especificidad', 'limite_unilateral_inferior_95'] >= MIN_SPECIFICITY,
    },
    {
        'criterio': 'VPN', 'regla': 'LI unilateral 95% ≥ 0.95',
        'valor_decisivo': metric_lookup.loc['vpn', 'limite_unilateral_inferior_95'],
        'cumple': metric_lookup.loc['vpn', 'limite_unilateral_inferior_95'] >= MIN_NPV,
    },
    {
        'criterio': 'Balance de selección', 'regla': 'máx. tamaño de efecto < 0.10',
        'valor_decisivo': max_balance_effect_observed, 'cumple': randomness_passed,
    },
])
standalone_equivalence = bool(decision_table.iloc[:4]['cumple'].all())
hybrid_first_pass_supported = bool(
    randomness_passed and decision_table.loc[decision_table['criterio'].isin(['Especificidad', 'VPN']), 'cumple'].all()
)

STRATUM_METRICS_FILE = OUTPUT_DIR / 'validacion_flash_pro_metricas_estratos.csv'
METRIC_CI_FILE = OUTPUT_DIR / 'validacion_flash_pro_metricas_bootstrap.csv'
PER_LABEL_METRICS_FILE = OUTPUT_DIR / 'validacion_flash_pro_metricas_etiquetas.csv'
DECISION_TABLE_FILE = OUTPUT_DIR / 'validacion_flash_pro_decisiones.csv'
stratum_metrics.to_csv(STRATUM_METRICS_FILE, index=False, encoding='utf-8')
metric_ci.to_csv(METRIC_CI_FILE, index=False, encoding='utf-8')
per_label_metrics.to_csv(PER_LABEL_METRICS_FILE, index=False, encoding='utf-8')
decision_table.to_csv(DECISION_TABLE_FILE, index=False, encoding='utf-8')

print(f'Videos (conglomerados): {len(video_ids):,}; réplicas bootstrap: {CLUSTER_BOOTSTRAP_REPS:,}')
display(stratum_metrics)
display(metric_ci)
display(per_label_metrics)
display(decision_table)
print(f'Equivalencia como etiquetador final autónomo: {standalone_equivalence}')
print(f'Uso respaldado como primera pasada de un flujo híbrido: {hybrid_first_pass_supported}')


Videos (conglomerados): 1,500; réplicas bootstrap: 5,000


,estrato,n,acuerdo_binario,prevalencia_dano_flash,prevalencia_dano_pro,coincidencia_exacta,jaccard
0,dirigida,4646,0.574473,0.998278,0.573181,0.325441,0.420620
1,control_seguro,5354,0.964326,0.000000,0.035674,0.930706,0.930706


,metrica,estimacion,ic95_inferior,ic95_superior,limite_unilateral_inferior_95,limite_unilateral_superior_95
0,accuracy_binaria,0.933262,0.927974,0.938592,0.928726,0.937637
1,sensibilidad,0.581563,0.544830,0.624197,0.549799,0.615783
2,especificidad,0.963223,0.961515,0.964799,0.961765,0.964532
3,vpp,0.573954,0.552844,0.593475,0.555694,0.590240
4,vpn,0.964312,0.958279,0.970239,0.959218,0.969259
5,coincidencia_exacta,0.882478,0.875053,0.889559,0.876236,0.888685
6,jaccard,0.890062,0.882742,0.897175,0.883888,0.896176
7,kappa_cohen,0.541503,0.517945,0.566059,0.521211,0.561828
8,ac1_gwet,0.921892,0.915228,0.928558,0.916236,0.927295
9,mcc,0.541517,0.518212,0.566300,0.521377,0.562122


,etiqueta,soporte_pro_estimado,precision,recall,f1
0,seguro,61893.927096,0.930955,0.965468,0.947898
1,seguro_ironia_marcada,2475.354875,0.777778,0.033955,0.065070
2,racismo_etnico_explicito,908.749080,0.525316,0.547102,0.535988
3,racismo_linguistico,94.778996,0.143939,0.240163,0.179998
4,clasismo_racial,661.902828,0.517520,0.347513,0.415811
5,discriminacion_regional,600.641301,0.553616,0.442794,0.492042
6,racismo_encubierto,348.948797,0.384259,0.284958,0.327241
7,misoginia_acoso_genero,1194.392432,0.556671,0.389178,0.458095
8,homofobia_transfobia,324.825884,0.599349,0.678627,0.636529
9,acoso_personal,1679.644624,0.380594,0.649065,0.479829


,criterio,regla,valor_decisivo,cumple
0,Equivalencia: desacuerdo binario,LS unilateral 95% ≤ 0.05,0.071274,False
1,Sensibilidad,LI unilateral 95% ≥ 0.90,0.549799,False
2,Especificidad,LI unilateral 95% ≥ 0.95,0.961765,True
3,VPN,LI unilateral 95% ≥ 0.95,0.959218,True
4,Balance de selección,máx. tamaño de efecto < 0.10,0.026638,True


Equivalencia como etiquetador final autónomo: False
Uso respaldado como primera pasada de un flujo híbrido: True


## 13.3 Resultados visuales

Las figuras se construyen directamente a partir de `selection_summary`, `stratum_metrics` y `metric_ci`; no contienen valores escritos manualmente. Además de mostrarse en el cuaderno, se guardan como PNG para que el resultado permanezca visible y pueda reutilizarse en el informe.

### 13.3.1 Exploración del diseño muestral y del daño observado

El panel izquierdo muestra qué fracción de cada estrato fue revisada por Pro. El derecho compara descriptivamente la proporción de daño asignada por Flash y Pro dentro de los casos revisados; estas barras no reemplazan la estimación poblacional ponderada.


In [38]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

def localizar_directorio_resultados_validacion() -> Path:
    if 'OUTPUT_DIR' in globals():
        return Path(globals()['OUTPUT_DIR'])
    for candidate in (Path.cwd(), *Path.cwd().parents):
        result_dir = candidate / 'datos' / 'etiquetado' / 'llm_api'
        if result_dir.exists():
            return result_dir
    raise FileNotFoundError('No se encontró datos/etiquetado/llm_api desde el directorio actual.')

OUTPUT_DIR = localizar_directorio_resultados_validacion()
SELECTION_SUMMARY_FILE = OUTPUT_DIR / 'validacion_flash_pro_diseno_muestral.csv'
STRATUM_METRICS_FILE = OUTPUT_DIR / 'validacion_flash_pro_metricas_estratos.csv'
if 'selection_summary' not in globals():
    if not SELECTION_SUMMARY_FILE.exists():
        raise RuntimeError(
            'Falta la tabla del diseño muestral. Ejecute una vez las celdas 13.1–13.2; '
            'después este gráfico funcionará también tras reiniciar el kernel.'
        )
    selection_summary = pd.read_csv(SELECTION_SUMMARY_FILE)
if 'stratum_metrics' not in globals():
    if not STRATUM_METRICS_FILE.exists():
        raise RuntimeError(
            'Falta la tabla de métricas por estrato. Ejecute una vez las celdas 13.1–13.2; '
            'después este gráfico funcionará también tras reiniciar el kernel.'
        )
    stratum_metrics = pd.read_csv(STRATUM_METRICS_FILE)

EXPLORATORY_FIGURE = OUTPUT_DIR / 'validacion_flash_pro_exploratoria.png'
stratum_labels = {'dirigida': 'Revisión dirigida', 'control_seguro': 'Control seguro'}
stratum_order = ['dirigida', 'control_seguro']
colors = {'Flash': '#4C78A8', 'Pro': '#F58518'}

selection_for_plot = selection_summary.set_index('estrato').loc[stratum_order]
metrics_for_plot = stratum_metrics.set_index('estrato').loc[stratum_order]
x = np.arange(len(stratum_order))
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), constrained_layout=True)

inclusion_rates = selection_for_plot['fraccion_final_revisada'].to_numpy()
bars = axes[0].bar(x, inclusion_rates, color=['#7A5195', '#2A9D8F'], width=0.62)
axes[0].set_title('Cobertura de la revisión Pro')
axes[0].set_ylabel('Porcentaje del estrato revisado')
axes[0].set_xticks(x, [stratum_labels[s] for s in stratum_order])
axes[0].set_ylim(0, 1)
axes[0].yaxis.set_major_formatter(PercentFormatter(1.0))
axes[0].grid(axis='y', alpha=0.25)
for bar, rate, (_, row) in zip(bars, inclusion_rates, selection_for_plot.iterrows()):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2, rate + 0.025,
        f"{rate:.1%}\n({int(row['n_revisado_pro']):,}/{int(row['N_flash']):,})",
        ha='center', va='bottom', fontsize=9,
    )

width = 0.34
flash_rates = metrics_for_plot['prevalencia_dano_flash'].to_numpy()
pro_rates = metrics_for_plot['prevalencia_dano_pro'].to_numpy()
flash_bars = axes[1].bar(x - width / 2, flash_rates, width, label='Flash', color=colors['Flash'])
pro_bars = axes[1].bar(x + width / 2, pro_rates, width, label='Pro', color=colors['Pro'])
axes[1].set_title('Daño observado en los casos revisados')
axes[1].set_ylabel('Chunks con al menos una etiqueta de daño')
axes[1].set_xticks(x, [stratum_labels[s] for s in stratum_order])
axes[1].set_ylim(0, 1.08)
axes[1].yaxis.set_major_formatter(PercentFormatter(1.0))
axes[1].legend(frameon=False)
axes[1].grid(axis='y', alpha=0.25)
for bar_group in (flash_bars, pro_bars):
    for bar in bar_group:
        axes[1].text(
            bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.025,
            f'{bar.get_height():.1%}', ha='center', va='bottom', fontsize=9,
        )

fig.suptitle('Exploración de la segunda pasada Flash → Pro', fontsize=14, fontweight='bold')
fig.savefig(EXPLORATORY_FIGURE, dpi=180, bbox_inches='tight', facecolor='white')
plt.close(fig)
print(f'Figura guardada en: {EXPLORATORY_FIGURE}')


Figura guardada en: D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\datos\etiquetado\llm_api\validacion_flash_pro_exploratoria.png


![Exploración del diseño muestral y las tasas de daño](../datos/etiquetado/llm_api/validacion_flash_pro_exploratoria.png)

**Lectura.** La revisión cubrió 83.5% del estrato dirigido, pero solo 8.3% del control seguro; esta asimetría explica la necesidad de ponderación. En el estrato dirigido Flash asignó daño casi siempre (99.8%), mientras que Pro lo hizo en 57.3%. En el control seguro Flash asignó 0% por definición, pero Pro encontró daño en 3.6%.

### 13.3.2 Inferencia y criterios de decisión

El siguiente *forest plot* presenta los cuatro criterios primarios en una escala común, donde valores mayores son mejores. Las líneas representan IC bootstrap bilaterales del 95%; la barra negra marca el límite inferior unilateral del 95% usado para decidir y el rombo señala el umbral preespecificado.


In [39]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

def localizar_directorio_resultados_inferencia() -> Path:
    if 'OUTPUT_DIR' in globals():
        return Path(globals()['OUTPUT_DIR'])
    for candidate in (Path.cwd(), *Path.cwd().parents):
        result_dir = candidate / 'datos' / 'etiquetado' / 'llm_api'
        if result_dir.exists():
            return result_dir
    raise FileNotFoundError('No se encontró datos/etiquetado/llm_api desde el directorio actual.')

OUTPUT_DIR = localizar_directorio_resultados_inferencia()
METRIC_CI_FILE = OUTPUT_DIR / 'validacion_flash_pro_metricas_bootstrap.csv'
if 'metric_ci' not in globals():
    if not METRIC_CI_FILE.exists():
        raise RuntimeError(
            'Falta la tabla bootstrap. Ejecute una vez la celda de estimación de la sección 13.2; '
            'después este gráfico funcionará también tras reiniciar el kernel.'
        )
    metric_ci = pd.read_csv(METRIC_CI_FILE)
metric_lookup = metric_ci.set_index('metrica')
max_equivalent_disagreement = globals().get('MAX_EQUIVALENT_DISAGREEMENT', 0.05)
min_sensitivity = globals().get('MIN_SENSITIVITY', 0.90)
min_specificity = globals().get('MIN_SPECIFICITY', 0.95)
min_npv = globals().get('MIN_NPV', 0.95)

INFERENCE_FIGURE = OUTPUT_DIR / 'validacion_flash_pro_inferencia.png'
inference_specs = [
    ('accuracy_binaria', 'Acuerdo daño/seguro', 1 - max_equivalent_disagreement),
    ('sensibilidad', 'Sensibilidad', min_sensitivity),
    ('especificidad', 'Especificidad', min_specificity),
    ('vpn', 'Valor predictivo negativo', min_npv),
]
inference_rows = []
for metric, label, threshold in inference_specs:
    row = metric_lookup.loc[metric]
    lower_one_sided = row['limite_unilateral_inferior_95']
    inference_rows.append({
        'metrica': label, 'estimacion': row['estimacion'],
        'ic95_inferior': row['ic95_inferior'], 'ic95_superior': row['ic95_superior'],
        'limite_unilateral': lower_one_sided, 'umbral': threshold,
        'cumple': bool(lower_one_sided >= threshold),
    })
inference_plot_df = pd.DataFrame(inference_rows)

fig, ax = plt.subplots(figsize=(10.5, 5.2), constrained_layout=True)
y_positions = np.arange(len(inference_plot_df))[::-1]
for y, (_, row) in zip(y_positions, inference_plot_df.iterrows()):
    color = '#2A9D8F' if row['cumple'] else '#D1495B'
    ax.hlines(y, row['ic95_inferior'], row['ic95_superior'], color='#6B7280', linewidth=3, alpha=0.7)
    ax.scatter(row['estimacion'], y, s=95, color=color, edgecolor='white', linewidth=0.8, zorder=3)
    ax.scatter(row['limite_unilateral'], y, s=120, color='black', marker='|', linewidth=2.2, zorder=4)
    ax.scatter(row['umbral'], y, s=65, facecolor='white', edgecolor='#3F3F46', marker='D', linewidth=1.4, zorder=3)
    ax.text(
        1.002, y, f"{row['estimacion']:.1%}  {'CUMPLE' if row['cumple'] else 'NO CUMPLE'}",
        ha='left', va='center', color=color, fontweight='bold', fontsize=9, clip_on=False,
    )

ax.set_yticks(y_positions, inference_plot_df['metrica'])
ax.set_xlim(0.50, 1.00)
ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xlabel('Estimación ponderada e intervalo bootstrap')
ax.set_title('Inferencia frente a los criterios preespecificados', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.25)
ax.spines[['top', 'right', 'left']].set_visible(False)
ax.scatter([], [], s=95, color='#2A9D8F', label='Estimación: cumple')
ax.scatter([], [], s=95, color='#D1495B', label='Estimación: no cumple')
ax.scatter([], [], s=120, color='black', marker='|', label='Límite unilateral 95%')
ax.scatter([], [], s=65, facecolor='white', edgecolor='#3F3F46', marker='D', label='Umbral')
ax.legend(loc='lower left', frameon=False, ncol=2, fontsize=8)
fig.savefig(INFERENCE_FIGURE, dpi=180, bbox_inches='tight', facecolor='white')
plt.close(fig)
print(f'Figura guardada en: {INFERENCE_FIGURE}')


Figura guardada en: D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\datos\etiquetado\llm_api\validacion_flash_pro_inferencia.png


![Inferencia bootstrap frente a los criterios de decisión](../datos/etiquetado/llm_api/validacion_flash_pro_inferencia.png)

**Lectura.** Especificidad y VPN superan sus umbrales aun usando el límite unilateral conservador. El acuerdo no alcanza el 95% exigido por el margen de desacuerdo y la sensibilidad queda lejos del 90%. Visualmente se confirma la conclusión: Flash es útil para una primera pasada con control posterior, pero no es equivalente a Pro como etiquetador final autónomo.


## 13.4 Calibración de la confianza y umbral operativo de `needs_review`

### 13.4.1 Qué significa la confianza

`score_confianza` es una autoevaluación producida por Flash; no es automáticamente una probabilidad ni un intervalo de confianza estadístico. Si estuviera bien calibrada, entre las predicciones con confianza 0.90 aproximadamente 90% deberían coincidir con la referencia. La calibración se comprueba comparando cada nivel declarado con su frecuencia empírica de acierto (Guo et al., 2017). Aquí se evalúan dos desenlaces: coincidencia exacta del conjunto de etiquetas y acuerdo binario daño/seguro, siempre respecto de Pro y no de una verdad humana.

El uso de `needs_review` se formula como **clasificación selectiva**: el sistema acepta automáticamente una fracción de casos (cobertura) y deriva el resto a revisión para reducir el riesgo entre los aceptados. No existe un umbral óptimo sin especificar el intercambio entre cobertura y error; por eso se estudia la curva riesgo–cobertura (El-Yaniv & Wiener, 2010).

### 13.4.2 Políticas comparadas y criterio de selección

Se comparan dos políticas reproducibles para cada valor observado de confianza $t$:

1. **Solo puntaje:** revisar si `score_confianza < t`.
2. **Alerta o puntaje:** revisar si Flash ya indicó `needs_review=True` **o** si `score_confianza < t`.

El criterio primario elige, dentro de cada política, el umbral con **máxima cobertura automática** cuyo límite inferior unilateral bootstrap del 95% sea al menos 90% para coincidencia exacta y 95% para acuerdo daño/seguro. Los pesos por estrato corrigen la selección dirigida de Pro y el bootstrap vuelve a muestrear videos completos. También se informa qué proporción de los desacuerdos termina efectivamente en revisión.

Estos márgenes son decisiones operativas y este análisis es exploratorio: el umbral se selecciona y evalúa con la misma muestra Pro. Debe congelarse y confirmarse en una muestra prospectiva independiente —preferentemente humana— antes de considerarlo una garantía de producción.


In [40]:
# Cálculo autónomo: puede ejecutarse después de reiniciar el kernel.
from collections import Counter
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

THRESHOLD_BOOTSTRAP_REPS = 5_000
THRESHOLD_ANALYSIS_SEED = 20_260_726
MIN_EXACT_ACCEPTED = 0.90
MIN_BINARY_ACCEPTED = 0.95
SAFE_CONFIDENCE_LABELS = {'seguro', 'seguro_ironia_marcada'}

def localizar_raiz_umbral() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'datos' / 'processed' / 'chunks_para_etiquetar.jsonl').exists():
            return candidate
    raise FileNotFoundError('No se encontró la raíz del proyecto desde el directorio actual.')

def leer_jsonl_umbral(path: Path) -> list[dict]:
    with path.open(encoding='utf-8') as file:
        return [json.loads(line) for line in file if line.strip()]

THRESHOLD_ROOT = localizar_raiz_umbral()
THRESHOLD_OUTPUT_DIR = THRESHOLD_ROOT / 'datos' / 'etiquetado' / 'llm_api'
threshold_flash_rows = leer_jsonl_umbral(
    THRESHOLD_OUTPUT_DIR / 'deepseek-v4-flash_labeled_chunks_seed42.jsonl'
)
threshold_pro_rows = leer_jsonl_umbral(
    THRESHOLD_OUTPUT_DIR / 'deepseek-v4-pro_revision_de_deepseek-v4-flash_seed42.jsonl'
)
threshold_chunks = {
    row['chunk_id']: row for row in leer_jsonl_umbral(
        THRESHOLD_ROOT / 'datos' / 'processed' / 'chunks_para_etiquetar.jsonl'
    )
}
threshold_flash_by_id = {row['chunk_id']: row for row in threshold_flash_rows}

def dano_umbral(row: dict) -> bool:
    return bool(set(row.get('labels', [])) - SAFE_CONFIDENCE_LABELS)

def estrato_umbral(row: dict) -> int:
    return 0 if row.get('needs_review') or dano_umbral(row) else 1

confidence_values = np.asarray(sorted({
    float(row['score_confianza']) for row in threshold_flash_rows
}))
population_by_stratum = np.bincount(
    [estrato_umbral(row) for row in threshold_flash_rows], minlength=2
).astype(float)
reviewed_by_stratum = np.bincount([
    estrato_umbral(threshold_flash_by_id[row['chunk_id']]) for row in threshold_pro_rows
], minlength=2).astype(float)
design_weight = population_by_stratum / reviewed_by_stratum

threshold_pair_rows = []
for pro_row in threshold_pro_rows:
    flash_row = threshold_flash_by_id[pro_row['chunk_id']]
    stratum = estrato_umbral(flash_row)
    threshold_pair_rows.append({
        'chunk_id': pro_row['chunk_id'],
        'video_id': threshold_chunks[pro_row['chunk_id']]['video_id'],
        'stratum': stratum,
        'weight': design_weight[stratum],
        'confidence': float(flash_row['score_confianza']),
        'flash_needs_review': bool(flash_row.get('needs_review')),
        'exact_agreement': set(flash_row.get('labels', [])) == set(pro_row.get('labels', [])),
        'binary_agreement': dano_umbral(flash_row) == dano_umbral(pro_row),
    })
threshold_pairs_df = pd.DataFrame(threshold_pair_rows)

# Calibración descriptiva por valor exacto del score.
population_confidence_counts = Counter(float(row['score_confianza']) for row in threshold_flash_rows)
calibration_rows = []
for confidence, group in threshold_pairs_df.groupby('confidence'):
    calibration_rows.append({
        'score_confianza': confidence,
        'N_flash': population_confidence_counts[confidence],
        'n_revisado_pro': len(group),
        'acuerdo_exacto': np.average(group['exact_agreement'], weights=group['weight']),
        'acuerdo_binario': np.average(group['binary_agreement'], weights=group['weight']),
    })
confidence_calibration = pd.DataFrame(calibration_rows).sort_values('score_confianza')
total_pair_weight = threshold_pairs_df['weight'].sum()
ece_exact = sum(
    row.N_flash / len(threshold_flash_rows) * abs(row.acuerdo_exacto - row.score_confianza)
    for row in confidence_calibration.itertuples()
)
ece_binary = sum(
    row.N_flash / len(threshold_flash_rows) * abs(row.acuerdo_binario - row.score_confianza)
    for row in confidence_calibration.itertuples()
)
brier_exact = np.average(
    (threshold_pairs_df['confidence'] - threshold_pairs_df['exact_agreement'].astype(float)) ** 2,
    weights=threshold_pairs_df['weight'],
)
brier_binary = np.average(
    (threshold_pairs_df['confidence'] - threshold_pairs_df['binary_agreement'].astype(float)) ** 2,
    weights=threshold_pairs_df['weight'],
)

# Contribuciones por video, estrato, política, umbral y estadístico.
# Estadísticos: n, aceptado, exacto_aceptado, binario_aceptado, error_exacto,
# error_exacto_revisado, error_binario, error_binario_revisado.
threshold_video_ids = sorted(threshold_pairs_df['video_id'].unique(), key=str)
threshold_video_index = {video_id: index for index, video_id in enumerate(threshold_video_ids)}
threshold_contributions = np.zeros((
    len(threshold_video_ids), 2, 2, len(confidence_values), 8
), dtype=float)
for pair in threshold_pair_rows:
    for policy_index in range(2):
        accepted = pair['confidence'] >= confidence_values
        if policy_index == 1:
            accepted = accepted & (not pair['flash_needs_review'])
        values = np.column_stack([
            np.ones(len(confidence_values)),
            accepted,
            accepted * pair['exact_agreement'],
            accepted * pair['binary_agreement'],
            np.full(len(confidence_values), not pair['exact_agreement']),
            (~accepted) * (not pair['exact_agreement']),
            np.full(len(confidence_values), not pair['binary_agreement']),
            (~accepted) * (not pair['binary_agreement']),
        ])
        threshold_contributions[
            threshold_video_index[pair['video_id']], pair['stratum'], policy_index
        ] += values

threshold_coverage = np.zeros((2, len(confidence_values)))
for policy_index in range(2):
    for threshold_index, threshold in enumerate(confidence_values):
        threshold_coverage[policy_index, threshold_index] = np.mean([
            row['score_confianza'] >= threshold
            and (policy_index == 0 or not row.get('needs_review'))
            for row in threshold_flash_rows
        ])

def metricas_umbral(sum_by_stratum: np.ndarray) -> np.ndarray:
    scale = (population_by_stratum / sum_by_stratum[:, 0, 0, 0])[:, None, None, None]
    totals = (sum_by_stratum * scale).sum(axis=0)
    exact_agreement = totals[:, :, 2] / totals[:, :, 1]
    binary_agreement = totals[:, :, 3] / totals[:, :, 1]
    captured_exact_errors = totals[:, :, 5] / totals[:, :, 4]
    captured_binary_errors = totals[:, :, 7] / totals[:, :, 6]
    return np.stack([
        exact_agreement, binary_agreement, captured_exact_errors, captured_binary_errors
    ], axis=-1)

threshold_point = metricas_umbral(threshold_contributions.sum(axis=0))
threshold_rng = np.random.default_rng(THRESHOLD_ANALYSIS_SEED)
threshold_bootstrap = np.empty((
    THRESHOLD_BOOTSTRAP_REPS, 2, len(confidence_values), 4
))
cluster_probability = np.full(len(threshold_video_ids), 1 / len(threshold_video_ids))
for replicate in range(THRESHOLD_BOOTSTRAP_REPS):
    multiplicity = threshold_rng.multinomial(len(threshold_video_ids), cluster_probability)
    sampled_sum = np.tensordot(multiplicity, threshold_contributions, axes=(0, 0))
    threshold_bootstrap[replicate] = metricas_umbral(sampled_sum)
threshold_lower = np.nanpercentile(threshold_bootstrap, 5, axis=0)
threshold_upper = np.nanpercentile(threshold_bootstrap, 95, axis=0)

threshold_rows = []
for policy_index, policy_name in enumerate(('solo_score', 'alerta_o_score')):
    for threshold_index, threshold in enumerate(confidence_values):
        threshold_rows.append({
            'politica': policy_name,
            'umbral': threshold,
            'cobertura_automatica': threshold_coverage[policy_index, threshold_index],
            'tasa_revision': 1 - threshold_coverage[policy_index, threshold_index],
            'acuerdo_exacto': threshold_point[policy_index, threshold_index, 0],
            'li_unilateral_exacto_95': threshold_lower[policy_index, threshold_index, 0],
            'acuerdo_binario': threshold_point[policy_index, threshold_index, 1],
            'li_unilateral_binario_95': threshold_lower[policy_index, threshold_index, 1],
            'captura_errores_exactos': threshold_point[policy_index, threshold_index, 2],
            'captura_errores_binarios': threshold_point[policy_index, threshold_index, 3],
        })
threshold_results = pd.DataFrame(threshold_rows)

optimal_rows = []
for policy_name, group in threshold_results.groupby('politica', sort=False):
    eligible = group[
        (group['li_unilateral_exacto_95'] >= MIN_EXACT_ACCEPTED)
        & (group['li_unilateral_binario_95'] >= MIN_BINARY_ACCEPTED)
    ]
    if eligible.empty:
        continue
    chosen = eligible.sort_values(
        ['cobertura_automatica', 'umbral'], ascending=[False, True]
    ).iloc[0]
    optimal_rows.append(chosen.to_dict())
optimal_thresholds = pd.DataFrame(optimal_rows)

THRESHOLD_RESULTS_FILE = THRESHOLD_OUTPUT_DIR / 'validacion_flash_pro_umbral_confianza.csv'
CALIBRATION_RESULTS_FILE = THRESHOLD_OUTPUT_DIR / 'validacion_flash_pro_calibracion_confianza.csv'
OPTIMAL_THRESHOLDS_FILE = THRESHOLD_OUTPUT_DIR / 'validacion_flash_pro_umbral_optimo.csv'
threshold_results.to_csv(THRESHOLD_RESULTS_FILE, index=False, encoding='utf-8')
confidence_calibration.to_csv(CALIBRATION_RESULTS_FILE, index=False, encoding='utf-8')
optimal_thresholds.to_csv(OPTIMAL_THRESHOLDS_FILE, index=False, encoding='utf-8')

print(f'ECE exacto={ece_exact:.4f}; ECE binario={ece_binary:.4f}; Brier exacto={brier_exact:.4f}; Brier binario={brier_binary:.4f}')
display(confidence_calibration[confidence_calibration['N_flash'] >= 100])
display(threshold_results[threshold_results['umbral'].isin([0.70, 0.75, 0.80, 0.85, 0.90, 0.92, 0.95, 0.97, 1.00])])
display(optimal_thresholds)


ECE exacto=0.0481; ECE binario=0.0292; Brier exacto=0.0900; Brier binario=0.0524


,score_confianza,N_flash,n_revisado_pro,acuerdo_exacto,acuerdo_binario
3,0.65,2786,2333,0.276897,0.491642
5,0.75,525,434,0.336406,0.470046
6,0.80,138,96,0.482661,0.622957
7,0.85,2534,1722,0.479730,0.741922
8,0.90,2632,236,0.807108,0.893542
10,0.95,60414,5052,0.936265,0.968321
12,1.00,673,55,0.981818,0.981818


,politica,umbral,cobertura_automatica,tasa_revision,acuerdo_exacto,li_unilateral_exacto_95,acuerdo_binario,li_unilateral_binario_95,captura_errores_exactos,captura_errores_binarios
4,solo_score,0.70,0.959973,0.040027,0.907864,0.901458,0.951804,0.946964,0.247505,0.306837
5,solo_score,0.75,0.959014,0.040986,0.908502,0.902112,0.952236,0.947396,0.253488,0.313775
6,solo_score,0.80,0.951498,0.048502,0.912978,0.906432,0.956009,0.951068,0.295517,0.372881
7,solo_score,0.85,0.949523,0.050477,0.913864,0.907343,0.956695,0.951714,0.304128,0.383932
8,solo_score,0.90,0.913246,0.086754,0.931449,0.924850,0.965395,0.960280,0.467753,0.526860
9,solo_score,0.92,0.875567,0.124433,0.936820,0.930435,0.968498,0.963729,0.529765,0.587128
10,solo_score,0.95,0.875553,0.124447,0.936808,0.930422,0.968492,0.963726,0.529765,0.587128
11,solo_score,0.97,0.010680,0.989320,0.983051,0.950820,0.983051,0.950820,0.998537,0.997424
14,solo_score,1.00,0.009635,0.990365,0.981818,0.947368,0.981818,0.947368,0.998537,0.997424
19,alerta_o_score,0.70,0.921063,0.078937,0.930260,0.923669,0.963927,0.958738,0.453419,0.502143


,politica,umbral,cobertura_automatica,tasa_revision,acuerdo_exacto,li_unilateral_exacto_95,acuerdo_binario,li_unilateral_binario_95,captura_errores_exactos,captura_errores_binarios
0,solo_score,0.8,0.951498,0.048502,0.912978,0.906432,0.956009,0.951068,0.295517,0.372881
1,alerta_o_score,0.3,0.921063,0.078937,0.930260,0.923669,0.963927,0.958738,0.453419,0.502143


In [41]:
# Curva riesgo–cobertura; carga el CSV si la celda de cálculo se ejecutó en otra sesión.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

if 'threshold_results' not in globals():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        threshold_csv = candidate / 'datos' / 'etiquetado' / 'llm_api' / 'validacion_flash_pro_umbral_confianza.csv'
        if threshold_csv.exists():
            threshold_results = pd.read_csv(threshold_csv)
            THRESHOLD_OUTPUT_DIR = threshold_csv.parent
            break
    else:
        raise RuntimeError('Ejecute primero la celda 13.4.2 para generar la tabla de umbrales.')

THRESHOLD_FIGURE = Path(THRESHOLD_OUTPUT_DIR) / 'validacion_flash_pro_riesgo_cobertura.png'
policy_style = {
    'solo_score': ('Solo score', '#4C78A8'),
    'alerta_o_score': ('Alerta o score', '#F58518'),
}
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), constrained_layout=True)
panels = [
    ('acuerdo_exacto', 'li_unilateral_exacto_95', 0.90, 'Coincidencia exacta'),
    ('acuerdo_binario', 'li_unilateral_binario_95', 0.95, 'Acuerdo daño/seguro'),
]
for axis, (estimate_col, lower_col, target, title) in zip(axes, panels):
    for policy, (label, color) in policy_style.items():
        group = threshold_results[threshold_results['politica'] == policy].sort_values('cobertura_automatica')
        axis.plot(group['cobertura_automatica'], group[estimate_col], color=color, marker='o', ms=4, label=f'{label}: estimación')
        axis.plot(group['cobertura_automatica'], group[lower_col], color=color, linestyle='--', alpha=0.8, label=f'{label}: límite 95%')
        for threshold in (0.80, 0.90, 0.95):
            row = group[np.isclose(group['umbral'], threshold)]
            if not row.empty and policy == 'solo_score':
                point = row.iloc[0]
                axis.annotate(
                    f't={threshold:.2f}', (point['cobertura_automatica'], point[estimate_col]),
                    xytext=(4, 6), textcoords='offset points', fontsize=8, color=color,
                )
    axis.axhline(target, color='#444444', linewidth=1.2, linestyle=':', label=f'Criterio {target:.0%}')
    axis.set_title(title)
    axis.set_xlabel('Cobertura automática')
    axis.set_ylabel('Acuerdo con Pro')
    axis.xaxis.set_major_formatter(PercentFormatter(1.0))
    axis.yaxis.set_major_formatter(PercentFormatter(1.0))
    axis.set_xlim(0, 1.01)
    axis.grid(alpha=0.25)
axes[0].set_ylim(0.85, 1.005)
axes[1].set_ylim(0.90, 1.005)
handles, labels = axes[1].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, -0.08), ncol=3, frameon=False, fontsize=8)
fig.suptitle('Curva riesgo–cobertura para decidir needs_review', fontsize=14, fontweight='bold')
fig.savefig(THRESHOLD_FIGURE, dpi=180, bbox_inches='tight', facecolor='white')
plt.close(fig)
print(f'Figura guardada en: {THRESHOLD_FIGURE}')


Figura guardada en: d:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\datos\etiquetado\llm_api\validacion_flash_pro_riesgo_cobertura.png


![Curva riesgo-cobertura del umbral de confianza](../datos/etiquetado/llm_api/validacion_flash_pro_riesgo_cobertura.png)

### 13.4.3 Resultados

Flash concentró 60,414 de 69,853 chunks (86.5%) exactamente en `score_confianza=0.95`. En ese nivel, Pro coincidió en el conjunto exacto de etiquetas en 93.63% y en daño/seguro en 96.83%. En cambio, para `score_confianza=0.90`, los acuerdos fueron 80.71% y 89.35%. Por tanto, el puntaje ordena el riesgo de manera útil, pero **no está perfectamente calibrado** ni debe leerse literalmente como una probabilidad. El error de calibración esperado ponderado fue 4.81 puntos porcentuales para coincidencia exacta y 2.92 para acuerdo binario.

Con la política **solo puntaje**, el umbral que maximiza cobertura y satisface los criterios primarios es **0.80**: se aceptarían automáticamente 95.15% de los chunks y se revisarían 4.85%. El acuerdo exacto estimado entre los aceptados es 91.30% (límite inferior unilateral 95%: 90.64%) y el acuerdo daño/seguro es 95.60% (límite inferior: 95.11%). Sin embargo, esa revisión capturaría solo 29.55% de los desacuerdos exactos y 37.29% de los binarios.

La alerta explícita actual de Flash es más selectiva que un corte puro: `needs_review=True` deriva 7.89% del corpus; entre los casos no derivados, el acuerdo exacto es 93.03% (límite inferior: 92.37%) y el binario 96.39% (límite inferior: 95.87%). La alerta captura 45.34% de los desacuerdos exactos y 50.21% de los binarios. Esto demuestra que Flash **sí identifica una parte importante de sus propias dudas**, pero deja sin señalar aproximadamente la mitad de sus desacuerdos con Pro.

Añadir la regla `score_confianza < 0.90` a la alerta existente produce una política conservadora: `needs_review_final = needs_review_flash OR score_confianza < 0.90`. La cobertura baja ligeramente a 91.24%; el acuerdo exacto sube a 93.19% (límite inferior: 92.53%) y el binario a 96.55% (límite inferior: 96.04%). La captura de desacuerdos aumenta a 47.18% y 52.84%, respectivamente. La mejora sobre la alerta sola es pequeña, por lo que el corte 0.90 se recomienda solamente cuando el costo adicional de revisar cerca de 0.86% del corpus sea aceptable.

### 13.4.4 Conclusión operativa

- Si se desea **máxima automatización** bajo los criterios 90%/95%, puede usarse `score_confianza < 0.80` para activar revisión.
- Si se desea **detectar mejor los errores**, debe conservarse la señal explícita `needs_review`; no conviene reemplazarla por el score.
- Como regla conservadora se propone `needs_review_final = needs_review_flash OR score_confianza < 0.90`, manteniendo además el control aleatorio de casos no señalados, porque ninguna regla capturó todos los desacuerdos.

No se recomienda elevar el umbral a 0.97 para perseguir 95% de coincidencia exacta: aunque alcanzaría ese límite, dejaría apenas 1.07% de cobertura automática y se apoya en pocos casos revisados. La recomendación 0.90 debe validarse prospectivamente y no convierte a Pro en verdad de terreno.


### 13.4.5 Aplicar el umbral sin volver a etiquetar con Flash

La siguiente celda crea un manifiesto derivado con la regla recomendada `needs_review_flash OR score_confianza < 0.90`. No altera el JSONL original de Flash. También cruza los identificadores con la salida Pro existente para separar los casos ya revisados de los pendientes y calcular el número máximo de llamadas con lotes de cinco.

Con los archivos actuales, la regla deriva 6,116 chunks: 4,695 ya cuentan con anotación Pro y quedan 1,421 pendientes. Recalcular el umbral y generar el manifiesto requiere **cero llamadas**; completar con Pro todos los pendientes requeriría como máximo 285 llamadas de cinco chunks, sin contar reintentos.

Por seguridad, `EJECUTAR_PRO_RECALIBRADO=False`: al ejecutarla normalmente **no llama a ninguna API**. Para solicitar a Pro únicamente los casos pendientes, primero deben ejecutarse las celdas de configuración, prompt, cliente y ejecutor (secciones 1, 3, 4, 5 y 7), cambiar el interruptor a `True` y ejecutar de nuevo solo esta celda. La salida Pro se escribe en un archivo nuevo y reanudable.


In [42]:
# Esta celda NO vuelve a etiquetar con Flash y NO llama a Pro salvo activación explícita.
from collections import defaultdict
from datetime import datetime
from pathlib import Path
import json
import math
import pandas as pd

UMBRAL_RECALIBRADO = 0.90
EJECUTAR_PRO_RECALIBRADO = True
BATCH_RECALIBRADO = int(globals().get('BATCH_SIZE', 5))

def localizar_raiz_aplicacion() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'datos' / 'processed' / 'chunks_para_etiquetar.jsonl').exists():
            return candidate
    raise FileNotFoundError('No se encontró la raíz del proyecto desde el directorio actual.')

def leer_jsonl_aplicacion(path: Path) -> list[dict]:
    with path.open(encoding='utf-8') as file:
        return [json.loads(line) for line in file if line.strip()]

APPLICATION_ROOT = localizar_raiz_aplicacion()
APPLICATION_OUTPUT_DIR = APPLICATION_ROOT / 'datos' / 'etiquetado' / 'llm_api'
application_flash_file = APPLICATION_OUTPUT_DIR / 'deepseek-v4-flash_labeled_chunks_seed42.jsonl'
application_pro_file = APPLICATION_OUTPUT_DIR / 'deepseek-v4-pro_revision_de_deepseek-v4-flash_seed42.jsonl'
application_flash_rows = leer_jsonl_aplicacion(application_flash_file)
application_pro_rows = leer_jsonl_aplicacion(application_pro_file)
application_existing_pro_ids = {row['chunk_id'] for row in application_pro_rows}

recalibrated_review_ids = [
    row['chunk_id'] for row in application_flash_rows
    if bool(row.get('needs_review')) or float(row.get('score_confianza', 0.0)) < UMBRAL_RECALIBRADO
]
recalibrated_already_reviewed_ids = [
    chunk_id for chunk_id in recalibrated_review_ids if chunk_id in application_existing_pro_ids
]
recalibrated_pending_ids = [
    chunk_id for chunk_id in recalibrated_review_ids if chunk_id not in application_existing_pro_ids
]
estimated_new_pro_calls = math.ceil(len(recalibrated_pending_ids) / BATCH_RECALIBRADO)

RECALIBRATED_MANIFEST_FILE = APPLICATION_OUTPUT_DIR / 'flash_needs_review_recalibrado_t090_seed42.manifest.json'
recalibrated_manifest = {
    'created_at': datetime.now().astimezone().isoformat(timespec='seconds'),
    'source_flash_file': str(application_flash_file),
    'existing_pro_file': str(application_pro_file),
    'rule': 'needs_review_flash OR score_confianza < threshold',
    'threshold': UMBRAL_RECALIBRADO,
    'flash_total': len(application_flash_rows),
    'review_total_recalibrated': len(recalibrated_review_ids),
    'already_reviewed_by_pro': len(recalibrated_already_reviewed_ids),
    'pending_new_pro': len(recalibrated_pending_ids),
    'batch_size': BATCH_RECALIBRADO,
    'estimated_new_pro_calls_without_retries': estimated_new_pro_calls,
    'review_ids': recalibrated_review_ids,
    'already_reviewed_ids': recalibrated_already_reviewed_ids,
    'pending_ids': recalibrated_pending_ids,
}
RECALIBRATED_MANIFEST_FILE.write_text(
    json.dumps(recalibrated_manifest, ensure_ascii=False, indent=2), encoding='utf-8'
)
routing_summary = pd.DataFrame([{
    'umbral': UMBRAL_RECALIBRADO,
    'regla': 'alerta original O score < umbral',
    'total_flash': len(application_flash_rows),
    'total_derivado_pro': len(recalibrated_review_ids),
    'ya_revisado_pro': len(recalibrated_already_reviewed_ids),
    'pendiente_pro': len(recalibrated_pending_ids),
    'batch_size': BATCH_RECALIBRADO,
    'llamadas_nuevas_estimadas': estimated_new_pro_calls,
}])
display(routing_summary)
print(f'Manifiesto guardado en: {RECALIBRATED_MANIFEST_FILE}')
print('Llamadas API realizadas por esta ejecución:', 0 if not EJECUTAR_PRO_RECALIBRADO else 'ver resultado del ejecutor')

if EJECUTAR_PRO_RECALIBRADO:
    required_runtime_names = [
        'ejecutar_etiquetado', 'REVIEW_MODEL_ID', 'REVIEW_ANNOTATOR_ID'
    ]
    missing_runtime_names = [name for name in required_runtime_names if name not in globals()]
    if missing_runtime_names:
        raise RuntimeError(
            'Antes de activar llamadas, ejecute las secciones 1, 3, 4, 5 y 7. '
            f'Faltan: {missing_runtime_names}'
        )
    canonical_rows = leer_jsonl_aplicacion(
        APPLICATION_ROOT / 'datos' / 'processed' / 'chunks_para_etiquetar.jsonl'
    )
    canonical_position = {row['chunk_id']: index for index, row in enumerate(canonical_rows)}
    rows_by_video = defaultdict(list)
    for row in canonical_rows:
        rows_by_video[row.get('video_id')].append(row)
    enriched_by_id = {}
    for video_rows in rows_by_video.values():
        video_rows.sort(key=lambda row: (row.get('start_seconds') or 0, canonical_position[row['chunk_id']]))
        for index, row in enumerate(video_rows):
            enriched = dict(row)
            if index > 0:
                enriched['contexto_anterior'] = video_rows[index - 1]['text']
            if index + 1 < len(video_rows):
                enriched['contexto_posterior'] = video_rows[index + 1]['text']
            enriched_by_id[row['chunk_id']] = enriched
    pending_records = [enriched_by_id[chunk_id] for chunk_id in recalibrated_pending_ids]
    recalibrated_pro_output = (
        APPLICATION_OUTPUT_DIR / 'deepseek-v4-pro_revision_umbral_recalibrado_t090_seed42.jsonl'
    )
    recalibrated_pro_stats = ejecutar_etiquetado(
        pending_records, recalibrated_pro_output, REVIEW_MODEL_ID, REVIEW_ANNOTATOR_ID,
        batch_size=BATCH_RECALIBRADO, limit=None,
    )
    display(recalibrated_pro_stats)


,umbral,regla,total_flash,total_derivado_pro,ya_revisado_pro,pendiente_pro,batch_size,llamadas_nuevas_estimadas
0,0.9,alerta original O score < umbral,69853,6116,4695,1421,5,285


Manifiesto guardado en: d:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\datos\etiquetado\llm_api\flash_needs_review_recalibrado_t090_seed42.manifest.json
Llamadas API realizadas por esta ejecución: ver resultado del ejecutor


,grupo,metrica,valor,porcentaje
0,PROGRESO,completados,1421.0000,100.000
1,PROGRESO,pendientes,0.0000,0.000
2,CLASIFICACION,chunks_seguro,868.0000,61.084
3,CLASIFICACION,chunks_con_dano,553.0000,38.916
4,CALIDAD,needs_review,406.0000,28.571
5,CALIDAD,confianza_media,0.8304,NaN
6,ETIQUETA,seguro,744.0000,52.357
7,ETIQUETA,seguro_ironia_marcada,124.0000,8.726
8,ETIQUETA,racismo_etnico_explicito,118.0000,8.304
9,ETIQUETA,racismo_linguistico,8.0000,0.563


deepseek-v4-pro_revision_umbral_recalibrado_t090_seed42:   0%|          | 0/1421 [00:00<?, ?chunk/s]

Intento 1/6 inválido; se reintentan 5 registro(s): orden/IDs incorrectos: esperado=['mIfMMpQa1ZA_0044', 'Q-0Ats8fYFw_0018', 'puz0-iw6dzk_0001', 'OHuAniIeAqU_0019', 'DU5EhpIalhU_0019'], recibido=['mIfMMpQa1ZA_0044', 'Q-0Ats8YfFw_0018', 'puz0-iw6dzk_0001', 'OHuAniIeAqU_0019', 'DU5EhpIalhU_0019']
Intento 1/6 inválido; se reintentan 5 registro(s): orden/IDs incorrectos: esperado=['gLE7dOYCFqI_0121', 'VZ9BPJD_SBk_0009', 'ZBT1Zc_lysc_0006', '5R53jbgXVo8_0096', 'BHqPOSTt1gI_0079'], recibido=['gLE7dOYCFqI_0121', 'VZ9BPJD_SBPk_0009', 'ZBT1Zc_lysc_0006', '5R53jbgXVo8_0096', 'BHqPOSTt1gI_0079']
{
  "output": "d:\\trabajo_PLN\\Trabajo_PLN-MIA-Grupo4\\datos\\etiquetado\\llm_api\\deepseek-v4-pro_revision_umbral_recalibrado_t090_seed42.jsonl",
  "model": "deepseek-v4-pro",
  "new_rows": 1421,
  "total_rows": 1421,
  "elapsed_seconds": 79.88,
  "chunks_per_minute": 1067.416,
  "usage": {
    "prompt_tokens": 1692753,
    "completion_tokens": 179263,
    "total_tokens": 1872016,
    "prompt_cache_hit_t

{'output': 'd:\\trabajo_PLN\\Trabajo_PLN-MIA-Grupo4\\datos\\etiquetado\\llm_api\\deepseek-v4-pro_revision_umbral_recalibrado_t090_seed42.jsonl',
 'model': 'deepseek-v4-pro',
 'new_rows': 1421,
 'total_rows': 1421,
 'elapsed_seconds': 79.88,
 'chunks_per_minute': 1067.416,
 'usage': {'prompt_tokens': 1692753,
  'completion_tokens': 179263,
  'total_tokens': 1872016,
  'prompt_cache_hit_tokens': 960000,
  'prompt_cache_miss_tokens': 732753},
 'estimated_cost_usd_new_rows': 0.478186,
 'metrics_file': 'd:\\trabajo_PLN\\Trabajo_PLN-MIA-Grupo4\\datos\\etiquetado\\llm_api\\deepseek-v4-pro_revision_umbral_recalibrado_t090_seed42.metrics.json',
 'completed': 1421,
 'pending': 0,
 'progress_pct': 100.0,
 'safe_chunks': 868,
 'damage_chunks': 553,
 'needs_review': 406,
 'needs_review_pct': 28.571,
 'mean_confidence': 0.8304,
 'label_counts': {'seguro': 744,
  'seguro_ironia_marcada': 124,
  'racismo_etnico_explicito': 118,
  'racismo_linguistico': 8,
  'clasismo_racial': 76,
  'discriminacion_reg

## 13.5 Diseño de la validación humana

La comparación Flash–Pro evalúa consistencia entre modelos, pero no establece validez sustantiva. Para convertir la conclusión en evidencia académicamente más sólida se propone una muestra humana ciega, estratificada y con doble codificación. El umbral `needs_review_flash OR score_confianza < 0.90` debe quedar congelado antes de observar las etiquetas humanas.

### 13.5.1 Tamaño muestral y supuestos

Para una proporción esperada $p$, nivel de confianza del 95% y semiamplitud deseada $e$, se parte de la expresión de Cochran

$$n_0=\frac{z_{0.975}^2p(1-p)}{e^2}, \qquad n_{FPC}=\frac{n_0}{1+(n_0-1)/N},$$

donde la segunda expresión incorpora la corrección por población finita (Cochran, 1977). El tamaño se infla por un efecto de diseño provisional de 1.50 para contemplar la dependencia entre chunks de un mismo video; los efectos de diseño modifican el tamaño efectivo y deben estimarse nuevamente con los datos humanos (Zins & Burgard, 2020). Para sensibilidad y especificidad, la cantidad de casos positivos disponibles también depende de la prevalencia (Buderer, 1996).

La propuesta contiene una muestra probabilística de 2,600 chunks para inferencia global y un suplemento dirigido de 400 casos raros:

- 1,600 casos aceptados automáticamente: `needs_review=False` y confianza mayor o igual a 0.90.
- 1,000 casos derivados: alerta original o confianza menor que 0.90.
- 400 casos adicionales de etiquetas raras o críticas, analizados por separado o con su probabilidad de inclusión explícita.

Los primeros dos estratos se seleccionarán aleatoriamente con semilla y manifiesto, distribuidos en al menos 800–1,000 videos y, de ser posible, con un máximo aproximado de tres chunks por video. El suplemento dirigido no debe incorporarse ingenuamente a la estimación global.

### 13.5.2 Protocolo de anotación

Cada uno de los 3,000 chunks debe ser etiquetado de manera independiente por dos personas que desconozcan la predicción, confianza y decisión de Flash y Pro: 6,000 asignaciones humanas. Todos los desacuerdos deben adjudicarse por una tercera persona. Antes del estudio se recomienda una ronda separada de entrenamiento de 150–200 chunks que no forme parte de la evaluación.

Antes de la adjudicación se informarán acuerdo observado, coincidencia exacta, Jaccard, kappa y AC1 globales y por etiqueta. La medición explícita del acuerdo entre codificadores es un componente central de la anotación lingüística reproducible (Artstein & Poesio, 2008). Después de la adjudicación, la etiqueta humana consensuada se utilizará como referencia y se aplicarán los pesos de diseño y el bootstrap por video.

Para afirmaciones por categoría se buscarán al menos 100 casos humanos positivos por etiqueta. Si una clase rara no alcanza ese número, su desempeño se reportará como exploratorio o se revisará el universo de candidatos; la muestra de 3,000 respalda principalmente conclusiones globales, binarias y de las clases frecuentes.


In [43]:
# Cálculo reproducible del tamaño y precisión esperada de la validación humana.
from pathlib import Path
import json
import math
import numpy as np
import pandas as pd
from scipy.stats import norm
from IPython.display import display

HUMAN_CONFIDENCE_LEVEL = 0.95
HUMAN_DESIGN_EFFECT = 1.50
HUMAN_ACCEPTED_N = 1_600
HUMAN_REVIEW_N = 1_000
HUMAN_RARE_SUPPLEMENT_N = 400
HUMAN_CODERS_PER_CHUNK = 2
HUMAN_MIN_VIDEOS = 800
HUMAN_PREFERRED_VIDEOS = 1_000
HUMAN_MAX_CHUNKS_PER_VIDEO_TARGET = 3
z_human = norm.ppf(1 - (1 - HUMAN_CONFIDENCE_LEVEL) / 2)

def localizar_raiz_validacion_humana() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'datos' / 'etiquetado' / 'llm_api').exists():
            return candidate
    raise FileNotFoundError('No se encontró la raíz del proyecto desde el directorio actual.')

HUMAN_ROOT = localizar_raiz_validacion_humana()
HUMAN_OUTPUT_DIR = HUMAN_ROOT / 'datos' / 'etiquetado' / 'llm_api'
threshold_table_file = HUMAN_OUTPUT_DIR / 'validacion_flash_pro_umbral_confianza.csv'
metric_table_file = HUMAN_OUTPUT_DIR / 'validacion_flash_pro_metricas_bootstrap.csv'
if not threshold_table_file.exists() or not metric_table_file.exists():
    raise RuntimeError('Ejecute primero las celdas estadísticas de las secciones 13.2 y 13.4.')
human_threshold_table = pd.read_csv(threshold_table_file)
human_metric_table = pd.read_csv(metric_table_file).set_index('metrica')
human_policy_row = human_threshold_table[
    (human_threshold_table['politica'] == 'alerta_o_score')
    & np.isclose(human_threshold_table['umbral'], 0.90)
].iloc[0]

HUMAN_POPULATION_N = 69_853
accepted_population = int(round(HUMAN_POPULATION_N * human_policy_row['cobertura_automatica']))
review_population = HUMAN_POPULATION_N - accepted_population
accepted_exact = float(human_policy_row['acuerdo_exacto'])
accepted_binary = float(human_policy_row['acuerdo_binario'])
global_exact = float(human_metric_table.loc['coincidencia_exacta', 'estimacion'])
global_binary = float(human_metric_table.loc['accuracy_binaria', 'estimacion'])
coverage_accepted = accepted_population / HUMAN_POPULATION_N
coverage_review = review_population / HUMAN_POPULATION_N
review_exact = (global_exact - coverage_accepted * accepted_exact) / coverage_review
review_binary = (global_binary - coverage_accepted * accepted_binary) / coverage_review

def fpc_variance_factor(population_n: int, sample_n: int) -> float:
    return (population_n - sample_n) / (population_n - 1)

def margin_stratum(proportion: float, population_n: int, sample_n: int) -> float:
    variance = (
        proportion * (1 - proportion) / sample_n
        * fpc_variance_factor(population_n, sample_n)
        * HUMAN_DESIGN_EFFECT
    )
    return z_human * math.sqrt(variance)

def margin_stratified(proportion_accepted: float, proportion_review: float) -> float:
    variance = HUMAN_DESIGN_EFFECT * (
        coverage_accepted ** 2
        * proportion_accepted * (1 - proportion_accepted) / HUMAN_ACCEPTED_N
        * fpc_variance_factor(accepted_population, HUMAN_ACCEPTED_N)
        + coverage_review ** 2
        * proportion_review * (1 - proportion_review) / HUMAN_REVIEW_N
        * fpc_variance_factor(review_population, HUMAN_REVIEW_N)
    )
    return z_human * math.sqrt(variance)

def required_sample_approx(proportion: float, population_n: int, half_width: float) -> int:
    n0 = z_human ** 2 * proportion * (1 - proportion) / half_width ** 2
    n_fpc = n0 / (1 + (n0 - 1) / population_n)
    return math.ceil(n_fpc * HUMAN_DESIGN_EFFECT)

human_validation_plan = pd.DataFrame([
    {
        'componente': 'Aceptado automáticamente', 'N': accepted_population,
        'n_humano': HUMAN_ACCEPTED_N, 'seleccion': 'aleatoria estratificada por video',
        'peso_base_N_sobre_n': accepted_population / HUMAN_ACCEPTED_N,
    },
    {
        'componente': 'Derivado por alerta o score', 'N': review_population,
        'n_humano': HUMAN_REVIEW_N, 'seleccion': 'aleatoria estratificada por video',
        'peso_base_N_sobre_n': review_population / HUMAN_REVIEW_N,
    },
    {
        'componente': 'Suplemento raro/crítico', 'N': np.nan,
        'n_humano': HUMAN_RARE_SUPPLEMENT_N, 'seleccion': 'dirigida; analizar por separado',
        'peso_base_N_sobre_n': np.nan,
    },
])

human_precision = pd.DataFrame([
    {
        'estimando': 'Coincidencia exacta global', 'proporcion_planificacion': global_exact,
        'margen_95_estimado': margin_stratified(accepted_exact, review_exact),
    },
    {
        'estimando': 'Acuerdo daño/seguro global', 'proporcion_planificacion': global_binary,
        'margen_95_estimado': margin_stratified(accepted_binary, review_binary),
    },
    {
        'estimando': 'Daño/desacuerdo omitido entre aceptados',
        'proporcion_planificacion': 1 - accepted_binary,
        'margen_95_estimado': margin_stratum(1 - accepted_binary, accepted_population, HUMAN_ACCEPTED_N),
    },
    {
        'estimando': 'Desacuerdo binario entre derivados',
        'proporcion_planificacion': 1 - review_binary,
        'margen_95_estimado': margin_stratum(1 - review_binary, review_population, HUMAN_REVIEW_N),
    },
])

human_sample_targets = pd.DataFrame([
    {
        'objetivo': 'Coincidencia exacta entre aceptados', 'semiamplitud_objetivo': 0.015,
        'n_aproximado_con_DEFF': required_sample_approx(accepted_exact, accepted_population, 0.015),
        'n_propuesto': HUMAN_ACCEPTED_N,
    },
    {
        'objetivo': 'Daño omitido entre aceptados', 'semiamplitud_objetivo': 0.011,
        'n_aproximado_con_DEFF': required_sample_approx(1 - accepted_binary, accepted_population, 0.011),
        'n_propuesto': HUMAN_ACCEPTED_N,
    },
    {
        'objetivo': 'Desacuerdo binario entre derivados', 'semiamplitud_objetivo': 0.036,
        'n_aproximado_con_DEFF': required_sample_approx(1 - review_binary, review_population, 0.036),
        'n_propuesto': HUMAN_REVIEW_N,
    },
])

human_total_chunks = HUMAN_ACCEPTED_N + HUMAN_REVIEW_N + HUMAN_RARE_SUPPLEMENT_N
human_annotation_assignments = human_total_chunks * HUMAN_CODERS_PER_CHUNK
HUMAN_PLAN_FILE = HUMAN_OUTPUT_DIR / 'plan_validacion_humana.csv'
HUMAN_PRECISION_FILE = HUMAN_OUTPUT_DIR / 'precision_esperada_validacion_humana.csv'
HUMAN_PLAN_JSON = HUMAN_OUTPUT_DIR / 'plan_validacion_humana.json'
human_validation_plan.to_csv(HUMAN_PLAN_FILE, index=False, encoding='utf-8')
human_precision.to_csv(HUMAN_PRECISION_FILE, index=False, encoding='utf-8')
HUMAN_PLAN_JSON.write_text(json.dumps({
    'confidence_level': HUMAN_CONFIDENCE_LEVEL,
    'provisional_design_effect': HUMAN_DESIGN_EFFECT,
    'probability_sample_chunks': HUMAN_ACCEPTED_N + HUMAN_REVIEW_N,
    'rare_targeted_supplement_chunks': HUMAN_RARE_SUPPLEMENT_N,
    'total_unique_chunks': human_total_chunks,
    'independent_coders_per_chunk': HUMAN_CODERS_PER_CHUNK,
    'human_annotation_assignments': human_annotation_assignments,
    'minimum_videos': HUMAN_MIN_VIDEOS,
    'preferred_videos': HUMAN_PREFERRED_VIDEOS,
    'target_max_chunks_per_video': HUMAN_MAX_CHUNKS_PER_VIDEO_TARGET,
    'threshold_frozen': 0.90,
    'rule_frozen': 'needs_review_flash OR score_confianza < 0.90',
}, ensure_ascii=False, indent=2), encoding='utf-8')

print(f'Muestra humana total: {human_total_chunks:,} chunks; asignaciones independientes: {human_annotation_assignments:,}')
display(human_validation_plan)
display(human_sample_targets)
display(human_precision)


Muestra humana total: 3,000 chunks; asignaciones independientes: 6,000


,componente,N,n_humano,seleccion,peso_base_N_sobre_n
0,Aceptado automáticamente,63737.0,1600,aleatoria estratificada por video,39.835625
1,Derivado por alerta o score,6116.0,1000,aleatoria estratificada por video,6.116000
2,Suplemento raro/crítico,NaN,400,dirigida; analizar por separado,NaN


,objetivo,semiamplitud_objetivo,n_aproximado_con_DEFF,n_propuesto
0,Coincidencia exacta entre aceptados,0.015,1598,1600
1,Daño omitido entre aceptados,0.011,1562,1600
2,Desacuerdo binario entre derivados,0.036,958,1000


,estimando,proporcion_planificacion,margen_95_estimado
0,Coincidencia exacta global,0.882478,0.013930
1,Acuerdo daño/seguro global,0.933262,0.010311
2,Daño/desacuerdo omitido entre aceptados,0.034520,0.010817
3,Desacuerdo binario entre derivados,0.402493,0.034050


### 13.5.3 Precisión esperada y alcance

La muestra propuesta comprende **3,000 chunks únicos**: 2,600 seleccionados probabilísticamente y 400 casos raros dirigidos. Con dos anotadores independientes se requieren **6,000 asignaciones humanas**, más la adjudicación de todos los desacuerdos. La ronda de entrenamiento de 150–200 chunks debe ser adicional y no incorporarse a las métricas finales.

Usando las tasas Flash–Pro solo como valores de planificación y un efecto de diseño provisional de 1.50, la muestra probabilística produciría márgenes aproximados del 95% cercanos a ±1.4 puntos porcentuales para coincidencia exacta global, ±1.1 puntos para acuerdo daño/seguro global y ±1.1 puntos para el daño/desacuerdo omitido entre casos aceptados. En el estrato derivado, el margen para desacuerdo binario sería aproximadamente ±3.5 puntos. Las celdas anteriores recalculan los valores exactos desde los CSV de resultados.

El tamaño de 3,000 es defendible para la conclusión global y para evaluar el mecanismo de derivación. No garantiza precisión individual para todas las clases raras: cualquier etiqueta con menos de 100 positivos humanos debe presentarse con intervalos amplios y alcance exploratorio.


## 13.6 Resultados e interpretación

La reconstrucción exacta del diseño confirmó 69,853 chunks en Flash y 10,000 revisiones Pro. Antes del tope hubo 12,032 candidatos: 5,566 de revisión dirigida y 6,466 controles seguros sorteados. El tope conservó 4,646 y 5,354, respectivamente. Las probabilidades finales de inclusión fueron 83.47% en el estrato dirigido y 8.33% en el control seguro; por eso los resultados poblacionales que siguen están ponderados, y no corresponden al promedio ingenuo de los 10,000 casos.

La auditoría del control seguro reproduce la semilla, el orden y los identificadores del manifiesto. Entre los 64,287 chunks elegibles, 5,354 quedaron en la revisión final. Las diferencias estandarizadas absolutas para longitud en caracteres, longitud en palabras, inicio y duración fueron 0.0083, 0.0066, 0.0266 y 0.0043. Las V de Cramér para canal, cuartil del orden canónico y 20 particiones deterministas del hash fueron aproximadamente 0.0210, 0.0103 y 0.0167. Todas están por debajo de 0.10. Esto respalda el balance observado de la parte aleatoria, aunque no convierte en aleatoria la mitad dirigida ni demuestra ausencia de sesgo en variables no observadas.

Después de postestratificar, el acuerdo binario estimado es 93.33% (IC 95%: 92.80%–93.86%), pero el límite superior unilateral del desacuerdo es 7.13%, superior al margen de equivalencia de 5%. La sensibilidad de Flash frente a Pro es 58.16% (IC 95%: 54.48%–62.42%; límite inferior unilateral: 54.98%), mientras que la especificidad es 96.32% (IC 95%: 96.15%–96.48%; límite inferior unilateral: 96.18%) y el VPN es 96.43% (IC 95%: 95.83%–97.02%; límite inferior unilateral: 95.92%). Kappa es 0.542 y AC1 es 0.922; la diferencia de prevalencias Flash menos Pro es apenas 0.10 puntos porcentuales (IC 95%: −0.51 a 0.72), ejemplo de que prevalencias globales similares pueden ocultar desacuerdos individuales compensados.

En términos poblacionales se estiman cerca de 2,295 falsos negativos (3.28% de los chunks) y 2,367 falsos positivos (3.39%). En el control que Flash declaró seguro, Pro detectó daño en 3.57% (IC 95%: 2.97%–4.17%). En la revisión dirigida, 42.53% de los casos que Flash señaló como daño fueron considerados seguros por Pro. La coincidencia exacta multi-etiqueta ponderada es 88.25% (IC 95%: 87.51%–88.96%), Jaccard es 89.01% y F1 macro es 47.68%; la tabla generada arriba permite localizar las clases minoritarias con desempeño débil.

### Conclusión

Los datos **sí respaldan el uso de Flash como primera pasada escalable dentro de un procedimiento híbrido**, porque el control seleccionado está bien balanceado y se cumplen los umbrales preespecificados de especificidad y VPN. Deben mantenerse una muestra aleatoria periódica de los casos seguros y una revisión Pro o humana de los casos de riesgo, ambigüedad y clases minoritarias.

Los datos **no respaldan la equivalencia de Flash como etiquetador final autónomo ni como verdad de terreno**: fracasan el margen de desacuerdo y el umbral de sensibilidad. Esta conclusión calificada es la compatible con los resultados; afirmar equivalencia total sería más fuerte que la evidencia disponible.

Para otorgar rigor académico a la validez sustantiva se requiere la validación humana definida en la sección 13.5: **3,000 chunks únicos** —1,600 aceptados automáticamente, 1,000 derivados y 400 casos raros dirigidos—, con dos codificadores independientes por chunk (**6,000 asignaciones**) y adjudicación de todos los desacuerdos. Los 2,600 casos probabilísticos permiten inferencia global ponderada; los 400 casos raros deben analizarse por separado. Hasta completar esta validación, la evidencia respalda a Flash como herramienta operativa híbrida frente a Pro, no como etiqueta humana definitiva.

### Reproducibilidad

Las dos celdas anteriores leen exclusivamente los JSONL y manifiestos existentes, reconstruyen la selección desde la semilla 42, fijan la semilla analítica 20,260,725 y recalculan todas las tablas e intervalos. No realizan llamadas a ninguna API. Los criterios se encuentran al inicio de la sección y no se modifican a partir de los resultados.


## 13.7 Referencias (APA 7)

Artstein, R., & Poesio, M. (2008). Inter-coder agreement for computational linguistics. *Computational Linguistics, 34*(4), 555–596. https://doi.org/10.1162/coli.07-034-R2

Austin, P. C. (2009). Balance diagnostics for comparing the distribution of baseline covariates between treatment groups in propensity-score matched samples. *Statistics in Medicine, 28*(25), 3083–3107. https://doi.org/10.1002/sim.3697

Begg, C. B., & Greenes, R. A. (1983). Assessment of diagnostic tests when disease verification is subject to selection bias. *Biometrics, 39*(1), 207–215. https://doi.org/10.2307/2530820

Buderer, N. M. F. (1996). Statistical methodology: I. Incorporating the prevalence of disease into the sample size calculation for sensitivity and specificity. *Academic Emergency Medicine, 3*(9), 895–900. https://doi.org/10.1111/j.1553-2712.1996.tb03538.x

Cochran, W. G. (1977). *Sampling techniques* (3rd ed.). John Wiley & Sons.

Cohen, J. (1960). A coefficient of agreement for nominal scales. *Educational and Psychological Measurement, 20*(1), 37–46. https://doi.org/10.1177/001316446002000104

El-Yaniv, R., & Wiener, Y. (2010). On the foundations of noise-free selective classification. *Journal of Machine Learning Research, 11*(53), 1605–1641. https://jmlr.org/papers/v11/el-yaniv10a.html

Field, C. A., & Welsh, A. H. (2007). Bootstrapping clustered data. *Journal of the Royal Statistical Society: Series B (Statistical Methodology), 69*(3), 369–390. https://doi.org/10.1111/j.1467-9868.2007.00593.x

Gwet, K. L. (2008). Computing inter-rater reliability and its variance in the presence of high agreement. *British Journal of Mathematical and Statistical Psychology, 61*(1), 29–48. https://doi.org/10.1348/000711006X126600

Guo, C., Pleiss, G., Sun, Y., & Weinberger, K. Q. (2017). On calibration of modern neural networks. In D. Precup & Y. W. Teh (Eds.), *Proceedings of the 34th International Conference on Machine Learning* (Vol. 70, pp. 1321–1330). PMLR. https://proceedings.mlr.press/v70/guo17a.html

Horvitz, D. G., & Thompson, D. J. (1952). A generalization of sampling without replacement from a finite universe. *Journal of the American Statistical Association, 47*(260), 663–685. https://doi.org/10.1080/01621459.1952.10483446

Lakens, D. (2017). Equivalence tests: A practical primer for *t* tests, correlations, and meta-analyses. *Social Psychological and Personality Science, 8*(4), 355–362. https://doi.org/10.1177/1948550617697177

Saito, T., & Rehmsmeier, M. (2015). The precision-recall plot is more informative than the ROC plot when evaluating binary classifiers on imbalanced datasets. *PLOS ONE, 10*(3), e0118432. https://doi.org/10.1371/journal.pone.0118432

Sokolova, M., & Lapalme, G. (2009). A systematic analysis of performance measures for classification tasks. *Information Processing & Management, 45*(4), 427–437. https://doi.org/10.1016/j.ipm.2009.03.002

Zins, S., & Burgard, J. P. (2020). Considering interviewer and design effects when planning sample sizes. *Survey Methodology, 46*(1). Statistics Canada. https://www150.statcan.gc.ca/n1/pub/12-001-x/2020001/article/00005-eng.htm
